# Phase 6.1 Final Training Policy and Narrow Tuning

This notebook is the second stage of Phase 6.

Phase 6.0 confirmed that the leading candidate is:

- `hybrid_slice_no_hard_case_override_pruned`

It also confirmed the active standalone challengers:

- `histgb_pruned_v1_no_anchor_no_regime`
- `ridge_direct_full`

The goal of this notebook is **not** to search for new model families or new routing rules.

The goal is to:

1. choose one shared ML training policy
2. do narrow finalist-only tuning under that policy
3. re-rank the tuned finalists
4. decide which single model advances to final inference and submission checks

## Active finalists entering 06.1

- `hybrid_slice_no_hard_case_override_pruned`
- `histgb_pruned_v1_no_anchor_no_regime`
- `ridge_direct_full`

Benchmark retained for comparison:
- `structured_winner`

## Out of scope

This notebook does not:
- introduce new model families
- redesign hybrid routing logic
- reopen broad feature search
- perform final inference on the true test set

Those belong outside this notebook.


## Decision rule for 06.1

This notebook uses the following ranking logic.

### Primary ranking metric
- mean RMSE on `primary_realistic`

### Guardrail
- the selected model must remain acceptable on `stress_test`

### Tie-breaks
1. lower `max_two_protocols`
2. simpler model
3. more stable slice behavior
4. better fold consistency

### Simplicity bias rule
If the leading hybrid beats the best standalone challenger only by a very small amount, prefer the simpler standalone model.

Practical threshold:
- within `0.005` on `primary_realistic`
- and within `0.01` on `stress_test`

### Shared-policy rule
The default assumption is that all ML components will use one shared training policy.

Component-specific training policies should only be allowed if the evidence is unusually strong.


In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [2]:
def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        has_root_files = (candidate / "train.csv").exists() and (candidate / "test.csv").exists()
        has_data_files = (candidate / "data" / "train.csv").exists() and (candidate / "data" / "test.csv").exists()
        if has_root_files or has_data_files:
            return candidate
    raise FileNotFoundError("Could not locate project root.")


def resolve_data_path(project_root: Path, filename: str) -> Path:
    for path in [project_root / filename, project_root / "data" / filename]:
        if path.exists():
            return path
    raise FileNotFoundError(f"Could not find {filename} in project root or data/")


PROJECT_ROOT = find_project_root()

train = pd.read_csv(resolve_data_path(PROJECT_ROOT, "train.csv"), parse_dates=["date"])
test = pd.read_csv(resolve_data_path(PROJECT_ROOT, "test.csv"), parse_dates=["date"])

for df in (train, test):
    if "tau" not in df.columns:
        df["tau"] = df["maturity_days"] / 365.0

train_date_summary = (
    train.groupby("date")
    .agg(
        total_rows=("row_id", "size"),
        observed_rows=("iv_observed", lambda s: s.notna().sum()),
        missing_rows=("iv_observed", lambda s: s.isna().sum()),
        observed_ratio=("iv_observed", lambda s: s.notna().mean()),
    )
    .sort_index()
)

train_dates = train_date_summary.index.to_list()

N_FOLDS = 4
VAL_DATES_PER_FOLD = 5
TOTAL_VAL_DATES = N_FOLDS * VAL_DATES_PER_FOLD
val_start_idx = len(train_dates) - TOTAL_VAL_DATES

fold_plan = pd.DataFrame(
    [
        {
            "fold": fold_id + 1,
            "train_start": train_dates[0],
            "train_end": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD - 1],
            "n_train_dates": val_start_idx + fold_id * VAL_DATES_PER_FOLD,
            "val_start": train_dates[val_start_idx + fold_id * VAL_DATES_PER_FOLD],
            "val_end": train_dates[val_start_idx + (fold_id + 1) * VAL_DATES_PER_FOLD - 1],
            "n_val_dates": VAL_DATES_PER_FOLD,
        }
        for fold_id in range(N_FOLDS)
    ]
)

print("06.1 bootstrap check")
print("Project root:", PROJECT_ROOT)
print("Train shape:", train.shape)
print("Test shape:", test.shape)
display(fold_plan)


06.1 bootstrap check
Project root: /Users/sauravkrishna/Documents/projects/NQFO-Impilied-volatility-surface
Train shape: (11640, 10)
Test shape: (3960, 10)


,fold,train_start,train_end,n_train_dates,val_start,val_end,n_val_dates
0,1,2025-01-02,2025-04-18,77,2025-04-21,2025-04-25,5
1,2,2025-01-02,2025-04-25,82,2025-04-28,2025-05-02,5
2,3,2025-01-02,2025-05-02,87,2025-05-05,2025-05-09,5
3,4,2025-01-02,2025-05-09,92,2025-05-12,2025-05-16,5


In [3]:
phase61_finalist_registry = pd.DataFrame(
    [
        {
            "model": "hybrid_slice_no_hard_case_override_pruned",
            "family": "hybrid",
            "target_mode": "mixed",
            "routing_style": "slice_gated",
            "nonlinear_branch": "histgb_pruned_v1_no_anchor_no_regime",
            "role_in_061": "main_finalist",
            "what_is_tuned": "underlying Ridge and pruned HistGB branch hyperparameters",
            "what_is_fixed": "routing logic",
            "notes": "Confirmed Phase 6.0 leader.",
        },
        {
            "model": "histgb_pruned_v1_no_anchor_no_regime",
            "family": "tree_ml",
            "target_mode": "residual",
            "routing_style": "single_model",
            "nonlinear_branch": "pruned_histgb_residual",
            "role_in_061": "nonlinear_challenger",
            "what_is_tuned": "HistGB hyperparameters",
            "what_is_fixed": "pruned feature set",
            "notes": "Best standalone nonlinear challenger.",
        },
        {
            "model": "ridge_direct_full",
            "family": "linear_ml",
            "target_mode": "direct_iv",
            "routing_style": "single_model",
            "nonlinear_branch": "(none)",
            "role_in_061": "linear_challenger",
            "what_is_tuned": "alpha",
            "what_is_fixed": "feature set",
            "notes": "Best conservative linear challenger.",
        },
        {
            "model": "structured_winner",
            "family": "structured",
            "target_mode": "direct_iv",
            "routing_style": "single_model",
            "nonlinear_branch": "(none)",
            "role_in_061": "untuned_benchmark",
            "what_is_tuned": "(none)",
            "what_is_fixed": "entire model",
            "notes": "Retained as benchmark only.",
        },
    ]
)

print("06.1 finalist registry")
display(phase61_finalist_registry)


06.1 finalist registry


,model,family,target_mode,routing_style,nonlinear_branch,role_in_061,what_is_tuned,what_is_fixed,notes
0,hybrid_slice_no_hard_case_override_pruned,hybrid,mixed,slice_gated,histgb_pruned_v1_no_anchor_no_regime,main_finalist,underlying Ridge and pruned HistGB branch hype...,routing logic,Confirmed Phase 6.0 leader.
1,histgb_pruned_v1_no_anchor_no_regime,tree_ml,residual,single_model,pruned_histgb_residual,nonlinear_challenger,HistGB hyperparameters,pruned feature set,Best standalone nonlinear challenger.
2,ridge_direct_full,linear_ml,direct_iv,single_model,(none),linear_challenger,alpha,feature set,Best conservative linear challenger.
3,structured_winner,structured,direct_iv,single_model,(none),untuned_benchmark,(none),entire model,Retained as benchmark only.


In [4]:
phase61_training_policy_registry = pd.DataFrame(
    [
        {
            "training_policy": "primary_only",
            "description": "Train ML components using only pseudo-masked training rows built under primary_realistic.",
            "shared_policy": True,
            "status": "candidate",
        },
        {
            "training_policy": "stress_only",
            "description": "Train ML components using only pseudo-masked training rows built under stress_test.",
            "shared_policy": True,
            "status": "candidate",
        },
        {
            "training_policy": "union_shared",
            "description": "Train ML components on the union of primary_realistic and stress_test pseudo-masked rows.",
            "shared_policy": True,
            "status": "candidate",
        },
    ]
)

print("06.1 training-policy registry")
display(phase61_training_policy_registry)


06.1 training-policy registry


,training_policy,description,shared_policy,status
0,primary_only,Train ML components using only pseudo-masked t...,True,candidate
1,stress_only,Train ML components using only pseudo-masked t...,True,candidate
2,union_shared,Train ML components on the union of primary_re...,True,candidate


In [5]:
BUCKET_COLS = ["maturity_label", "option_type"]
NODE_COLS = ["maturity_label", "moneyness", "option_type"]

MASK_PROTOCOL_CONFIG = {
    "stress_test": {
        "base_hide_rate": 0.10,
        "node_weight": 1.00,
        "support_weight": 0.00,
    },
    "primary_realistic": {
        "base_hide_rate": 0.10,
        "node_weight": 0.65,
        "support_weight": 0.35,
    },
}

LOCKED_MASK_SEED = "nqfo-val-v1"
PHASE4_SHRINK_ALPHA = 0.25

OVERALL_TEST_MISSING_RATE = test["iv_observed"].isna().mean()

test_bucket_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(BUCKET_COLS)["is_missing"]
    .mean()
    .rename("test_bucket_missing_rate")
    .reset_index()
)

test_node_pattern = (
    test.assign(is_missing=test["iv_observed"].isna())
    .groupby(NODE_COLS)["is_missing"]
    .mean()
    .rename("test_node_missing_rate")
    .reset_index()
)

surface_levels = pd.concat(
    [
        train[["moneyness", "maturity_label", "maturity_days"]],
        test[["moneyness", "maturity_label", "maturity_days"]],
    ],
    ignore_index=True,
)

moneyness_levels = sorted(surface_levels["moneyness"].dropna().unique().tolist())
maturity_levels = (
    surface_levels[["maturity_label", "maturity_days"]]
    .drop_duplicates()
    .sort_values("maturity_days")["maturity_label"]
    .tolist()
)

m_idx = {m: i for i, m in enumerate(moneyness_levels)}
t_idx = {t: i for i, t in enumerate(maturity_levels)}


def stable_uniform(key: str) -> float:
    digest = hashlib.md5(key.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12 - 1)


def opposite_option(opt: str) -> str:
    return "put" if opt == "call" else "call"


def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(np.asarray(y_true, dtype=float), np.asarray(y_pred, dtype=float))))


def local_support_profile(target_rows: pd.DataFrame, visible_rows: pd.DataFrame) -> pd.DataFrame:
    prof = target_rows.copy()

    visible_key_set = set(
        zip(
            visible_rows["date"],
            visible_rows["moneyness"],
            visible_rows["maturity_label"],
            visible_rows["option_type"],
        )
    )

    opp_visible = []
    same_maturity_adj_count = []
    same_moneyness_adj_count = []

    for d, m, t, o in zip(
        prof["date"],
        prof["moneyness"],
        prof["maturity_label"],
        prof["option_type"],
    ):
        opp_visible.append((d, m, t, opposite_option(o)) in visible_key_set)

        i = m_idx[m]
        j = t_idx[t]

        same_maturity_candidates = []
        if i - 1 >= 0:
            same_maturity_candidates.append((d, moneyness_levels[i - 1], t, o))
        if i + 1 < len(moneyness_levels):
            same_maturity_candidates.append((d, moneyness_levels[i + 1], t, o))

        same_moneyness_candidates = []
        if j - 1 >= 0:
            same_moneyness_candidates.append((d, m, maturity_levels[j - 1], o))
        if j + 1 < len(maturity_levels):
            same_moneyness_candidates.append((d, m, maturity_levels[j + 1], o))

        same_maturity_adj_count.append(sum(c in visible_key_set for c in same_maturity_candidates))
        same_moneyness_adj_count.append(sum(c in visible_key_set for c in same_moneyness_candidates))

    prof["opp_option_visible"] = opp_visible
    prof["same_maturity_adj_visible_count"] = same_maturity_adj_count
    prof["same_moneyness_adj_visible_count"] = same_moneyness_adj_count
    prof["local_support_score"] = (
        prof["opp_option_visible"].astype(int)
        + prof["same_maturity_adj_visible_count"]
        + prof["same_moneyness_adj_visible_count"]
    )
    return prof


def build_masked_validation_rows_with_protocol(
    full_df: pd.DataFrame,
    target_dates,
    protocol_name: str,
    seed: str = LOCKED_MASK_SEED,
) -> pd.DataFrame:
    cfg = MASK_PROTOCOL_CONFIG[protocol_name]

    val_df = full_df.loc[full_df["date"].isin(target_dates)].copy()
    val_df["is_orig_observed"] = val_df["iv_observed"].notna()
    val_df["is_orig_missing"] = ~val_df["is_orig_observed"]

    val_df = val_df.merge(test_bucket_pattern, on=BUCKET_COLS, how="left")
    val_df = val_df.merge(test_node_pattern, on=NODE_COLS, how="left")

    val_df["bucket_hide_rate_on_observed"] = (
        cfg["base_hide_rate"] * val_df["test_bucket_missing_rate"] / OVERALL_TEST_MISSING_RATE
    )

    val_df["priority_noise"] = val_df.apply(
        lambda r: stable_uniform(
            f"{seed}|{protocol_name}|{r['date'].date()}|{r['maturity_label']}|{r['moneyness']:.4f}|{r['option_type']}"
        ),
        axis=1,
    )

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    observed_support = local_support_profile(observed_pool, observed_pool)[["row_id", "local_support_score"]]
    val_df = val_df.merge(observed_support, on="row_id", how="left")
    val_df["local_support_score"] = val_df["local_support_score"].fillna(0).astype(int)

    val_df["is_pseudo_hidden"] = False

    observed_pool = val_df.loc[val_df["is_orig_observed"]].copy()
    for _, g in observed_pool.groupby(["date", *BUCKET_COLS], sort=False):
        n_obs = len(g)
        n_hide = int(np.round(g["bucket_hide_rate_on_observed"].iloc[0] * n_obs))
        if n_hide <= 0:
            continue

        node_rank = g["test_node_missing_rate"].rank(method="average", pct=True)
        support_rank = g["local_support_score"].rank(method="average", pct=True)

        selection_priority = (
            cfg["node_weight"] * node_rank
            + cfg["support_weight"] * support_rank
            + 1e-6 * g["priority_noise"]
        )

        chosen_idx = (
            g.assign(selection_priority=selection_priority)
            .sort_values(["selection_priority", "row_id"], ascending=[False, True])
            .head(n_hide)
            .index
        )
        val_df.loc[chosen_idx, "is_pseudo_hidden"] = True

    val_df["is_scored_target"] = val_df["is_pseudo_hidden"]
    val_df["is_visible_anchor"] = val_df["is_orig_observed"] & ~val_df["is_pseudo_hidden"]
    val_df["is_effectively_missing"] = val_df["is_orig_missing"] | val_df["is_pseudo_hidden"]
    val_df["mask_protocol"] = protocol_name

    return val_df.sort_values(["date", "maturity_days", "option_type", "moneyness"]).reset_index(drop=True)


In [7]:
def build_node_lookup(observed_df: pd.DataFrame, pred_name: str) -> pd.DataFrame:
    return (
        observed_df.groupby(NODE_COLS)["iv_observed"]
        .median()
        .rename(pred_name)
        .reset_index()
    )


def predict_recent_node_median(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = target_rows.copy()

    observed_train = train_history.loc[train_history["iv_observed"].notna()].copy()
    global_median = observed_train["iv_observed"].median()

    recent_dates = sorted(observed_train["date"].unique())[-lookback_dates:]
    recent_obs = observed_train.loc[observed_train["date"].isin(recent_dates)].copy()

    recent_lookup = build_node_lookup(recent_obs, "recent_node_pred")
    full_lookup = build_node_lookup(observed_train, "full_node_pred")

    out = out.merge(recent_lookup, on=NODE_COLS, how="left")
    out = out.merge(full_lookup, on=NODE_COLS, how="left")

    out["iv_pred"] = out["recent_node_pred"].fillna(out["full_node_pred"]).fillna(global_median)
    out["pred_source"] = np.select(
        [
            out["recent_node_pred"].notna(),
            out["full_node_pred"].notna(),
        ],
        [
            "recent_node_median",
            "full_node_median",
        ],
        default="global_median",
    )
    return out


def predict_same_date_linear_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_recent_node_median(train_history, target_rows, lookback_dates=lookback_dates).copy()

    out["interp_pred"] = np.nan
    out["interp_source"] = pd.Series(index=out.index, dtype="object")

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        x = anchors["moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        if len(anchors) == 1:
            interp_vals = np.repeat(y[0], len(g))
            interp_label = "same_date_single_anchor"
        else:
            interp_vals = np.interp(g["moneyness"].to_numpy(), x, y)
            interp_label = "same_date_linear_interp"

        out.loc[g.index, "interp_pred"] = interp_vals
        out.loc[g.index, "interp_source"] = interp_label

    use_interp = out["interp_pred"].notna()
    out.loc[use_interp, "iv_pred"] = out.loc[use_interp, "interp_pred"]
    out["pred_source"] = np.where(use_interp, out["interp_source"], out["pred_source"])

    return out


def predict_quadratic_smile_logm(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    degree: int = 2,
    min_anchors: int = 5,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["log_moneyness"] = np.log(out["moneyness"])
    out["smile_pred"] = np.nan
    out["smile_anchor_count"] = 0
    out["pred_source_smile"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["log_moneyness", "iv_observed"]]
            .dropna()
            .sort_values("log_moneyness")
        )

        if len(anchors) < min_anchors:
            continue
        if anchors["log_moneyness"].nunique() < degree + 1:
            continue

        x = anchors["log_moneyness"].to_numpy()
        y = anchors["iv_observed"].to_numpy()

        x_center = x.mean()
        coeffs = np.polyfit(x - x_center, y, deg=degree)

        target_x = g["log_moneyness"].to_numpy()
        pred = np.polyval(coeffs, target_x - x_center)

        in_range = (target_x >= x.min()) & (target_x <= x.max())
        pred = np.where(in_range, pred, np.nan)
        pred = np.clip(pred, pred_lo, pred_hi)

        out.loc[g.index, "smile_pred"] = pred
        out.loc[g.index, "smile_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_smile"] = np.where(in_range, "quadratic_smile_logm", pd.NA)

    use_smile = out["smile_pred"].notna()
    out.loc[use_smile, "iv_pred"] = out.loc[use_smile, "smile_pred"]
    out["pred_source"] = np.where(use_smile, out["pred_source_smile"], out["pred_source"])

    return out


def predict_total_variance_maturity_interp(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    out = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    out["anchor_total_variance"] = np.where(
        out["iv_observed"].notna(),
        (out["iv_observed"] / 100.0) ** 2 * out["tau"],
        np.nan,
    )
    out["tv_pred"] = np.nan
    out["tv_anchor_count"] = 0
    out["pred_source_tv"] = pd.Series(index=out.index, dtype="object")

    observed_train = train_history.loc[train_history["iv_observed"].notna(), "iv_observed"]
    pred_lo = observed_train.quantile(0.001)
    pred_hi = observed_train.quantile(0.999)

    for (_, moneyness, option_type), g_idx in out.groupby(
        ["date", "moneyness", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["tau", "anchor_total_variance"]]
            .dropna()
            .sort_values("tau")
        )

        if len(anchors) < 2:
            continue

        x = anchors["tau"].to_numpy()
        y = anchors["anchor_total_variance"].to_numpy()
        target_tau = g["tau"].to_numpy()

        in_range = (target_tau >= x.min()) & (target_tau <= x.max())
        pred_tv = np.interp(target_tau, x, y)
        pred_tv = np.where(in_range, pred_tv, np.nan)
        pred_tv = np.where(pred_tv > 0, pred_tv, np.nan)

        pred_iv = 100.0 * np.sqrt(pred_tv / target_tau)
        pred_iv = np.clip(pred_iv, pred_lo, pred_hi)

        out.loc[g.index, "tv_pred"] = pred_iv
        out.loc[g.index, "tv_anchor_count"] = len(anchors)
        out.loc[g.index, "pred_source_tv"] = np.where(in_range, "tv_maturity_interp", pd.NA)

    use_tv = out["tv_pred"].notna()
    out.loc[use_tv, "iv_pred"] = out.loc[use_tv, "tv_pred"]
    out["pred_source"] = np.where(use_tv, out["pred_source_tv"], out["pred_source"])

    return out


def predict_structured_region_blend(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    smile_weight_center: float = 0.65,
    smile_weight_wing: float = 0.35,
) -> pd.DataFrame:
    base = predict_same_date_linear_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    smile = predict_quadratic_smile_logm(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "smile_pred"]].copy()

    tv = predict_total_variance_maturity_interp(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    )[["row_id", "tv_pred"]].copy()

    out = base.merge(smile, on="row_id", how="left")
    out = out.merge(tv, on="row_id", how="left")

    out["wing_center_bucket"] = np.where(
        out["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )

    both_available = out["smile_pred"].notna() & out["tv_pred"].notna()
    only_smile = out["smile_pred"].notna() & out["tv_pred"].isna()
    only_tv = out["tv_pred"].notna() & out["smile_pred"].isna()

    center_mask = both_available & (out["wing_center_bucket"] == "center")
    wing_mask = both_available & (out["wing_center_bucket"] == "wing")

    out.loc[center_mask, "iv_pred"] = (
        smile_weight_center * out.loc[center_mask, "smile_pred"]
        + (1.0 - smile_weight_center) * out.loc[center_mask, "tv_pred"]
    )
    out.loc[wing_mask, "iv_pred"] = (
        smile_weight_wing * out.loc[wing_mask, "smile_pred"]
        + (1.0 - smile_weight_wing) * out.loc[wing_mask, "tv_pred"]
    )
    out.loc[only_smile, "iv_pred"] = out.loc[only_smile, "smile_pred"]
    out.loc[only_tv, "iv_pred"] = out.loc[only_tv, "tv_pred"]

    out["pred_source"] = np.select(
        [
            center_mask,
            wing_mask,
            only_smile,
            only_tv,
        ],
        [
            "structured_region_blend_center",
            "structured_region_blend_wing",
            "quadratic_smile_only",
            "tv_maturity_only",
        ],
        default=out["pred_source"],
    )

    return out


def predict_structured_region_blend_callput_shrink(
    train_history: pd.DataFrame,
    target_rows: pd.DataFrame,
    lookback_dates: int = 20,
    shrink_alpha: float = PHASE4_SHRINK_ALPHA,
) -> pd.DataFrame:
    out = predict_structured_region_blend(
        train_history,
        target_rows,
        lookback_dates=lookback_dates,
    ).copy()

    opposite_visible = out.loc[
        out["is_visible_anchor"],
        ["date", "moneyness", "maturity_label", "option_type", "iv_observed"],
    ].copy()
    opposite_visible["option_type"] = opposite_visible["option_type"].map(opposite_option)
    opposite_visible = opposite_visible.rename(columns={"iv_observed": "opp_visible_iv"})

    out = out.merge(
        opposite_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    use_shrink = out["opp_visible_iv"].notna()
    out.loc[use_shrink, "iv_pred"] = (
        (1.0 - shrink_alpha) * out.loc[use_shrink, "iv_pred"]
        + shrink_alpha * out.loc[use_shrink, "opp_visible_iv"]
    )

    out["pred_source"] = np.where(
        use_shrink,
        "structured_region_blend_callput_shrink",
        out["pred_source"],
    )

    return out


In [9]:
ML_MIN_HISTORY_DATES = 20
ML_MASK_DATES_PER_FOLD = 20


def get_ml_candidate_dates_for_fold(
    fold_row,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
):
    fold_train_dates = train_date_summary.loc[:fold_row.train_end].index.to_list()
    eligible = fold_train_dates[min_history_dates:]
    return eligible[-n_mask_dates:]


def make_single_date_training_bundle(
    train_df: pd.DataFrame,
    target_date,
    protocol_name: str,
) -> dict:
    history_df = train_df.loc[train_df["date"] < target_date].copy()
    masked_target_date = build_masked_validation_rows_with_protocol(
        train_df,
        [target_date],
        protocol_name=protocol_name,
    )

    if history_df.empty:
        raise ValueError("History dataframe is empty. Choose a later target date.")

    if not (history_df["date"].max() < pd.Timestamp(target_date)):
        raise ValueError("Temporal leakage detected: history includes target date or later.")

    return {
        "target_date": pd.Timestamp(target_date),
        "protocol": protocol_name,
        "train_history": history_df,
        "masked_target_date": masked_target_date,
    }


def add_true_support_columns(scored_df: pd.DataFrame) -> pd.DataFrame:
    out = scored_df.copy()

    if "local_support_score" in out.columns:
        out = out.rename(columns={"local_support_score": "mask_local_support_score"})

    scored_targets = out.loc[out["is_scored_target"]].copy()
    visible_anchors = out.loc[out["is_visible_anchor"]].copy()

    support = local_support_profile(scored_targets, visible_anchors)[
        [
            "row_id",
            "opp_option_visible",
            "same_maturity_adj_visible_count",
            "same_moneyness_adj_visible_count",
            "local_support_score",
        ]
    ].copy()

    support = support.rename(columns={"local_support_score": "true_local_support_score"})
    support["any_local_same_date_support"] = (
        support["opp_option_visible"]
        | (support["same_maturity_adj_visible_count"] > 0)
        | (support["same_moneyness_adj_visible_count"] > 0)
    )
    support["hard_case"] = ~support["any_local_same_date_support"]

    out = out.merge(support, on="row_id", how="left")

    out["opp_option_visible"] = out["opp_option_visible"].astype("boolean").fillna(False).astype(bool)
    out["same_maturity_adj_visible_count"] = out["same_maturity_adj_visible_count"].fillna(0).astype(int)
    out["same_moneyness_adj_visible_count"] = out["same_moneyness_adj_visible_count"].fillna(0).astype(int)
    out["true_local_support_score"] = out["true_local_support_score"].fillna(0).astype(int)
    out["any_local_same_date_support"] = out["any_local_same_date_support"].astype("boolean").fillna(False).astype(bool)
    out["hard_case"] = out["hard_case"].astype("boolean").fillna(False).astype(bool)

    if "mask_local_support_score" in out.columns:
        out["mask_local_support_score"] = out["mask_local_support_score"].fillna(0).astype(int)

    return out


def add_same_maturity_anchor_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    out["left_anchor_iv"] = np.nan
    out["right_anchor_iv"] = np.nan
    out["left_anchor_dist"] = np.nan
    out["right_anchor_dist"] = np.nan
    out["same_maturity_visible_anchor_count"] = 0

    for (_, maturity_label, option_type), g_idx in out.groupby(
        ["date", "maturity_label", "option_type"], sort=False
    ).groups.items():
        g = out.loc[g_idx].copy()

        anchors = (
            g.loc[g["is_visible_anchor"], ["moneyness", "iv_observed"]]
            .dropna()
            .sort_values("moneyness")
        )

        if len(anchors) == 0:
            continue

        anchor_x = anchors["moneyness"].to_numpy()
        anchor_y = anchors["iv_observed"].to_numpy()
        target_x = g["moneyness"].to_numpy()

        left_iv = []
        right_iv = []
        left_dist = []
        right_dist = []

        for x in target_x:
            left_mask = anchor_x <= x
            right_mask = anchor_x >= x

            if left_mask.any():
                lx = anchor_x[left_mask][-1]
                ly = anchor_y[left_mask][-1]
                left_iv.append(ly)
                left_dist.append(abs(x - lx))
            else:
                left_iv.append(np.nan)
                left_dist.append(np.nan)

            if right_mask.any():
                rx = anchor_x[right_mask][0]
                ry = anchor_y[right_mask][0]
                right_iv.append(ry)
                right_dist.append(abs(rx - x))
            else:
                right_iv.append(np.nan)
                right_dist.append(np.nan)

        out.loc[g.index, "left_anchor_iv"] = left_iv
        out.loc[g.index, "right_anchor_iv"] = right_iv
        out.loc[g.index, "left_anchor_dist"] = left_dist
        out.loc[g.index, "right_anchor_dist"] = right_dist
        out.loc[g.index, "same_maturity_visible_anchor_count"] = len(anchors)

    return out


def add_same_node_opposite_option_feature(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()

    opp_visible = (
        out.loc[out["is_visible_anchor"], ["date", "moneyness", "maturity_label", "option_type", "iv_observed"]]
        .copy()
    )
    opp_visible["option_type"] = opp_visible["option_type"].map(opposite_option)
    opp_visible = opp_visible.rename(columns={"iv_observed": "opp_visible_iv_same_node"})

    out = out.merge(
        opp_visible,
        on=["date", "moneyness", "maturity_label", "option_type"],
        how="left",
    )

    return out


def get_nearest_moneyness_rows(df: pd.DataFrame, target: float) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    distances = (df["moneyness"] - target).abs()
    min_dist = distances.groupby(df["date"]).transform("min")
    return df.loc[min_dist.eq(distances)].copy()


def add_date_level_regime_proxy_features(masked_df: pd.DataFrame) -> pd.DataFrame:
    out = masked_df.copy()
    visible = out.loc[out["is_visible_anchor"] & out["iv_observed"].notna()].copy()

    total_rows = out.groupby("date").size().rename("date_total_row_count")
    visible_count = visible.groupby("date").size().rename("date_visible_anchor_count")
    visible_ratio = (visible_count / total_rows).rename("date_visible_anchor_ratio")
    visible_mean = visible.groupby("date")["iv_observed"].mean().rename("date_visible_iv_mean")
    visible_dispersion = visible.groupby("date")["iv_observed"].std().rename("date_visible_iv_dispersion")

    atm = (
        get_nearest_moneyness_rows(visible, 1.0)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_atm_iv_proxy")
    )
    left = (
        get_nearest_moneyness_rows(visible, 0.9)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_0p9_proxy")
    )
    right = (
        get_nearest_moneyness_rows(visible, 1.1)
        .groupby("date")["iv_observed"]
        .mean()
        .rename("date_iv_1p1_proxy")
    )

    maturity_means = (
        visible.groupby(["date", "maturity_days"])["iv_observed"]
        .mean()
        .reset_index()
        .sort_values(["date", "maturity_days"])
    )
    short_end = (
        maturity_means.groupby("date").first()["iv_observed"]
        .rename("date_short_end_iv_proxy")
    )
    long_end = (
        maturity_means.groupby("date").last()["iv_observed"]
        .rename("date_long_end_iv_proxy")
    )

    proxy_df = pd.concat(
        [
            total_rows,
            visible_count,
            visible_ratio,
            visible_mean,
            visible_dispersion,
            atm,
            left,
            right,
            short_end,
            long_end,
        ],
        axis=1,
    ).reset_index()

    proxy_df["date_visible_anchor_count"] = proxy_df["date_visible_anchor_count"].fillna(0).astype(int)
    proxy_df["date_visible_anchor_ratio"] = proxy_df["date_visible_anchor_ratio"].fillna(0.0)
    proxy_df["date_skew_proxy"] = proxy_df["date_iv_0p9_proxy"] - proxy_df["date_iv_1p1_proxy"]
    proxy_df["date_term_slope_proxy"] = (
        proxy_df["date_short_end_iv_proxy"] - proxy_df["date_long_end_iv_proxy"]
    )

    out = out.merge(proxy_df, on="date", how="left")
    return out


def build_feature_table_for_masked_bundle(
    train_history: pd.DataFrame,
    masked_target_rows: pd.DataFrame,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    base = masked_target_rows.copy()

    linear_pred = predict_same_date_linear_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred"]
    ].rename(columns={"iv_pred": "same_date_linear_interp_pred"})

    smile_pred = predict_quadratic_smile_logm(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "smile_pred"]
    ].rename(columns={"smile_pred": "quadratic_smile_logm_pred"})

    tv_pred = predict_total_variance_maturity_interp(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "tv_pred"]
    ].rename(columns={"tv_pred": "tv_maturity_interp_pred"})

    region_pred = predict_structured_region_blend(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "iv_pred", "pred_source"]
    ].rename(
        columns={
            "iv_pred": "structured_region_blend_pred",
            "pred_source": "structured_region_blend_source",
        }
    )

    winner_pred = predict_structured_region_blend_callput_shrink(
        train_history,
        base,
        lookback_dates=lookback_dates,
        shrink_alpha=PHASE4_SHRINK_ALPHA,
    )[["row_id", "iv_pred", "pred_source"]].rename(
        columns={
            "iv_pred": "structured_winner_pred",
            "pred_source": "structured_winner_source",
        }
    )

    node_pred = predict_recent_node_median(train_history, base, lookback_dates=lookback_dates)[
        ["row_id", "recent_node_pred", "full_node_pred"]
    ]

    feat = base.merge(linear_pred, on="row_id", how="left")
    feat = feat.merge(smile_pred, on="row_id", how="left")
    feat = feat.merge(tv_pred, on="row_id", how="left")
    feat = feat.merge(region_pred, on="row_id", how="left")
    feat = feat.merge(winner_pred, on="row_id", how="left")
    feat = feat.merge(node_pred, on="row_id", how="left")

    feat = add_true_support_columns(feat)
    feat = add_same_maturity_anchor_features(feat)
    feat = add_same_node_opposite_option_feature(feat)
    feat = add_date_level_regime_proxy_features(feat)

    feat["support_score_gap"] = feat["true_local_support_score"] - feat.get("mask_local_support_score", 0)
    feat["has_opp_same_node_visible"] = feat["opp_visible_iv_same_node"].notna().astype(int)

    feat["log_moneyness"] = np.log(feat["moneyness"])
    feat["is_call"] = (feat["option_type"] == "call").astype(int)
    feat["wing_center_bucket"] = np.where(
        feat["moneyness"].between(0.9, 1.1, inclusive="both"),
        "center",
        "wing",
    )
    feat["is_center"] = (feat["wing_center_bucket"] == "center").astype(int)
    feat["is_wing"] = (feat["wing_center_bucket"] == "wing").astype(int)
    feat["is_edge_maturity"] = feat["maturity_label"].isin(["1M", "6M"]).astype(int)
    feat["is_interior_maturity"] = 1 - feat["is_edge_maturity"]

    feat["smile_minus_linear"] = feat["quadratic_smile_logm_pred"] - feat["same_date_linear_interp_pred"]
    feat["tv_minus_linear"] = feat["tv_maturity_interp_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_linear"] = feat["structured_winner_pred"] - feat["same_date_linear_interp_pred"]
    feat["winner_minus_region"] = feat["structured_winner_pred"] - feat["structured_region_blend_pred"]

    feat["target_iv"] = feat["iv_observed"]
    feat["target_residual"] = feat["target_iv"] - feat["structured_winner_pred"]

    return feat


def build_training_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
    n_mask_dates: int = ML_MASK_DATES_PER_FOLD,
) -> pd.DataFrame:
    target_dates = get_ml_candidate_dates_for_fold(
        fold_row,
        min_history_dates=min_history_dates,
        n_mask_dates=n_mask_dates,
    )

    tables = []
    for target_date in target_dates:
        bundle = make_single_date_training_bundle(
            train_df,
            target_date=target_date,
            protocol_name=protocol_name,
        )

        feat = build_feature_table_for_masked_bundle(
            bundle["train_history"],
            bundle["masked_target_date"],
            lookback_dates=lookback_dates,
        )

        scored = feat.loc[feat["is_scored_target"]].copy()
        if not scored["is_orig_observed"].all():
            raise ValueError("Pseudo-masked training examples must come from originally observed rows only.")

        scored["target_date"] = pd.Timestamp(target_date)
        scored["protocol"] = protocol_name
        scored["fold"] = fold_row.fold
        scored["n_history_dates"] = bundle["train_history"]["date"].nunique()
        tables.append(scored)

    return pd.concat(tables, ignore_index=True)


def build_validation_table_for_fold_protocol(
    train_df: pd.DataFrame,
    fold_row,
    protocol_name: str,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    val_dates = train_date_summary.loc[fold_row.val_start:fold_row.val_end].index.to_list()
    train_history = train_df.loc[train_df["date"] <= fold_row.train_end].copy()
    masked_val_rows = build_masked_validation_rows_with_protocol(
        train_df,
        val_dates,
        protocol_name=protocol_name,
    )

    feat = build_feature_table_for_masked_bundle(
        train_history,
        masked_val_rows,
        lookback_dates=lookback_dates,
    )

    scored = feat.loc[feat["is_scored_target"]].copy()
    if not scored["is_orig_observed"].all():
        raise ValueError("Validation scored rows must come from originally observed rows only.")

    scored["target_date"] = scored["date"]
    scored["protocol"] = protocol_name
    scored["fold"] = fold_row.fold
    scored["n_history_dates"] = train_history["date"].nunique()
    return scored.reset_index(drop=True)


def summarize_feature_table(feature_table: pd.DataFrame) -> pd.Series:
    return pd.Series(
        {
            "n_rows": len(feature_table),
            "n_dates": feature_table["target_date"].nunique(),
            "date_min": feature_table["target_date"].min().date().isoformat(),
            "date_max": feature_table["target_date"].max().date().isoformat(),
            "target_iv_missing": int(feature_table["target_iv"].isna().sum()),
            "target_residual_missing": int(feature_table["target_residual"].isna().sum()),
            "winner_pred_missing": int(feature_table["structured_winner_pred"].isna().sum()),
            "mean_target_iv": feature_table["target_iv"].mean(),
            "mean_target_residual": feature_table["target_residual"].mean(),
            "rmse_structured_winner": rmse(feature_table["target_iv"], feature_table["structured_winner_pred"]),
        }
    )


def schema_alignment_report(train_table: pd.DataFrame, val_table: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    train_cols = set(train_table.columns)
    val_cols = set(val_table.columns)
    train_only = sorted(train_cols - val_cols)
    val_only = sorted(val_cols - train_cols)

    summary = pd.Series(
        {
            "n_train_cols": len(train_cols),
            "n_val_cols": len(val_cols),
            "n_common_cols": len(train_cols & val_cols),
            "n_train_only_cols": len(train_only),
            "n_val_only_cols": len(val_only),
            "schemas_match": len(train_only) == 0 and len(val_only) == 0,
        }
    )
    detail = pd.DataFrame(
        {
            "train_only_cols": pd.Series(train_only, dtype="object"),
            "val_only_cols": pd.Series(val_only, dtype="object"),
        }
    )
    return summary, detail


In [10]:
def rmse_from_series(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def prepare_design_matrices(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
):
    X_train = train_df.loc[:, feature_cols].copy()
    X_val = val_df.loc[:, feature_cols].copy()

    for df in (X_train, X_val):
        bool_cols = df.select_dtypes(include=["bool"]).columns.tolist()
        for col in bool_cols:
            df[col] = df[col].astype(int)

    cat_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X_train.columns if c not in cat_cols]

    if num_cols:
        train_num = X_train[num_cols].copy()
        val_num = X_val[num_cols].copy()

        medians = train_num.median(numeric_only=True)
        train_num = train_num.fillna(medians)
        val_num = val_num.fillna(medians)
    else:
        train_num = pd.DataFrame(index=X_train.index)
        val_num = pd.DataFrame(index=X_val.index)

    if cat_cols:
        train_cat = pd.get_dummies(X_train[cat_cols], dummy_na=True)
        val_cat = pd.get_dummies(X_val[cat_cols], dummy_na=True)
        train_cat, val_cat = train_cat.align(val_cat, join="outer", axis=1, fill_value=0)
    else:
        train_cat = pd.DataFrame(index=X_train.index)
        val_cat = pd.DataFrame(index=X_val.index)

    X_train_final = pd.concat([train_num, train_cat], axis=1)
    X_val_final = pd.concat([val_num, val_cat], axis=1)

    return X_train_final, X_val_final


def evaluate_ridge_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    alpha: float = 1.0,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = Ridge(alpha=alpha)
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "alpha": alpha,
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[["row_id", "target_date", "target_iv", "structured_winner_pred"]].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model, X_train.columns.tolist()


def evaluate_histgb_model(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    feature_cols,
    target_col: str,
    learning_rate: float = 0.05,
    max_depth: int = 3,
    max_iter: int = 300,
    min_samples_leaf: int = 10,
    prediction_mode: str = "direct_iv",
):
    X_train, X_val = prepare_design_matrices(train_df, val_df, feature_cols)
    y_train = train_df[target_col].copy()
    y_val = val_df["target_iv"].copy()

    model = HistGradientBoostingRegressor(
        learning_rate=learning_rate,
        max_depth=max_depth,
        max_iter=max_iter,
        min_samples_leaf=min_samples_leaf,
        random_state=42,
    )
    model.fit(X_train, y_train)

    raw_pred = model.predict(X_val)

    if prediction_mode == "residual":
        final_pred = val_df["structured_winner_pred"].to_numpy() + raw_pred
    elif prediction_mode == "direct_iv":
        final_pred = raw_pred
    else:
        raise ValueError(f"Unknown prediction_mode: {prediction_mode}")

    out = {
        "target_col": target_col,
        "prediction_mode": prediction_mode,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "max_iter": max_iter,
        "min_samples_leaf": min_samples_leaf,
        "n_features_after_encoding": X_train.shape[1],
        "train_target_mean": float(y_train.mean()),
        "val_rmse": rmse_from_series(y_val, final_pred),
        "structured_winner_rmse": rmse_from_series(val_df["target_iv"], val_df["structured_winner_pred"]),
    }

    pred_df = val_df[
        [
            "row_id",
            "target_date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "target_iv",
            "structured_winner_pred",
            "target_residual",
            "structured_winner_source",
        ]
    ].copy()
    pred_df["ml_raw_pred"] = raw_pred
    pred_df["iv_pred"] = final_pred

    return out, pred_df, model


In [11]:
FEATURE_GROUPS = {
    "row_local_core": [
        "moneyness",
        "log_moneyness",
        "maturity_days",
        "tau",
        "is_call",
        "is_center",
        "is_wing",
        "is_edge_maturity",
        "is_interior_maturity",
        "option_type",
        "maturity_label",
        "wing_center_bucket",
    ],
    "structured_predictions": [
        "structured_winner_pred",
        "structured_region_blend_pred",
        "tv_maturity_interp_pred",
        "quadratic_smile_logm_pred",
        "same_date_linear_interp_pred",
        "structured_winner_source",
        "structured_region_blend_source",
    ],
    "structured_gaps": [
        "smile_minus_linear",
        "tv_minus_linear",
        "winner_minus_linear",
        "winner_minus_region",
    ],
    "same_date_anchor_values": [
        "opp_visible_iv_same_node",
        "has_opp_same_node_visible",
        "left_anchor_iv",
        "right_anchor_iv",
        "left_anchor_dist",
        "right_anchor_dist",
        "same_maturity_visible_anchor_count",
    ],
    "support_geometry": [
        "opp_option_visible",
        "same_maturity_adj_visible_count",
        "same_moneyness_adj_visible_count",
        "true_local_support_score",
        "mask_local_support_score",
        "support_score_gap",
        "hard_case",
    ],
    "historical_priors": [
        "recent_node_pred",
        "full_node_pred",
    ],
    "date_regime_proxies": [
        "date_atm_iv_proxy",
        "date_skew_proxy",
        "date_term_slope_proxy",
        "date_visible_anchor_ratio",
        "date_visible_iv_dispersion",
        "date_visible_iv_mean",
        "date_visible_anchor_count",
    ],
}

FULL_HISTGB_ABLATION_FEATURES = sorted(
    {
        col
        for cols in FEATURE_GROUPS.values()
        for col in cols
    }
)


def feature_cols_from_groups(
    included_groups=None,
    excluded_groups=None,
) -> list[str]:
    included_groups = included_groups or list(FEATURE_GROUPS.keys())
    excluded_groups = excluded_groups or []

    included_cols = {
        col
        for group_name in included_groups
        for col in FEATURE_GROUPS[group_name]
    }
    excluded_cols = {
        col
        for group_name in excluded_groups
        for col in FEATURE_GROUPS[group_name]
    }

    final_cols = [col for col in FULL_HISTGB_ABLATION_FEATURES if col in included_cols and col not in excluded_cols]
    return final_cols


PHASE6_PRUNED_HISTGB_FEATURES = feature_cols_from_groups(
    included_groups=list(FEATURE_GROUPS.keys()),
    excluded_groups=["same_date_anchor_values", "date_regime_proxies"],
)

phase6_feature_registry = pd.DataFrame(
    [
        {
            "feature_set": "FULL_HISTGB_ABLATION_FEATURES",
            "n_raw_feature_cols": len(FULL_HISTGB_ABLATION_FEATURES),
            "notes": "Full standalone HistGB feature set from 05_1.",
        },
        {
            "feature_set": "PHASE6_PRUNED_HISTGB_FEATURES",
            "n_raw_feature_cols": len(PHASE6_PRUNED_HISTGB_FEATURES),
            "notes": "Pruned standalone nonlinear feature set: no same_date_anchor_values, no date_regime_proxies.",
        },
    ]
)

print("Phase 6 feature registry")
display(phase6_feature_registry)


Phase 6 feature registry


,feature_set,n_raw_feature_cols,notes
0,FULL_HISTGB_ABLATION_FEATURES,46,Full standalone HistGB feature set from 05_1.
1,PHASE6_PRUNED_HISTGB_FEATURES,32,Pruned standalone nonlinear feature set: no sa...


In [12]:
PROTOCOLS = ["primary_realistic", "stress_test"]


def build_training_table_for_policy_fold(
    train_df: pd.DataFrame,
    fold_row,
    training_policy: str,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    if training_policy == "primary_only":
        policy_protocols = ["primary_realistic"]
    elif training_policy == "stress_only":
        policy_protocols = ["stress_test"]
    elif training_policy == "union_shared":
        policy_protocols = ["primary_realistic", "stress_test"]
    else:
        raise ValueError(f"Unknown training_policy: {training_policy}")

    tables = []
    for protocol_name in policy_protocols:
        table = build_training_table_for_fold_protocol(
            train_df,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=lookback_dates,
        ).copy()
        table["training_policy"] = training_policy
        table["source_protocol_for_training_rows"] = protocol_name
        tables.append(table)

    return pd.concat(tables, ignore_index=True)


In [13]:
def build_prediction_panel_for_policy_eval(
    val_table: pd.DataFrame,
    structured_pred_df: pd.DataFrame,
    ridge_pred_df: pd.DataFrame,
    pruned_histgb_pred_df: pd.DataFrame,
) -> pd.DataFrame:
    base = val_table[
        [
            "row_id",
            "target_date",
            "target_iv",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "wing_center_bucket",
            "hard_case",
            "structured_winner_source",
        ]
    ].copy()

    structured_pred = structured_pred_df[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "structured_winner__iv_pred"}
    )
    ridge_pred = ridge_pred_df[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "ridge_direct_full__iv_pred"}
    )
    pruned_histgb_pred = pruned_histgb_pred_df[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "histgb_pruned_v1_no_anchor_no_regime__iv_pred"}
    )

    panel = base.merge(structured_pred, on="row_id", how="left")
    panel = panel.merge(ridge_pred, on="row_id", how="left")
    panel = panel.merge(pruned_histgb_pred, on="row_id", how="left")

    return panel


def apply_hybrid_slice_no_hard_case_override_pruned(panel_df: pd.DataFrame) -> pd.DataFrame:
    out = panel_df.copy()

    out["hybrid_model"] = "hybrid_slice_no_hard_case_override_pruned"
    out["hybrid_route"] = "histgb_pruned_v1_no_anchor_no_regime"
    out["iv_pred"] = out["histgb_pruned_v1_no_anchor_no_regime__iv_pred"]

    ridge_mask = out["structured_winner_source"] == "tv_maturity_only"
    out.loc[ridge_mask, "hybrid_route"] = "ridge_direct_full"
    out.loc[ridge_mask, "iv_pred"] = out.loc[ridge_mask, "ridge_direct_full__iv_pred"]

    structured_mask = (
        (out["wing_center_bucket"] == "center")
        | (out["structured_winner_source"] == "quadratic_smile_only")
    )
    out.loc[structured_mask, "hybrid_route"] = "structured_winner"
    out.loc[structured_mask, "iv_pred"] = out.loc[structured_mask, "structured_winner__iv_pred"]

    return out


In [14]:
fold1_train_policy_union = build_training_table_for_policy_fold(
    train,
    fold_plan.iloc[0],
    training_policy="union_shared",
    lookback_dates=20,
)

fold1_primary_val_table = build_validation_table_for_fold_protocol(
    train,
    fold_plan.iloc[0],
    protocol_name="primary_realistic",
    lookback_dates=20,
)

print("06.1 union_shared Fold 1 training table summary")
display(summarize_feature_table(fold1_train_policy_union).to_frame("value"))

print("06.1 Fold 1 primary_realistic validation table summary")
display(summarize_feature_table(fold1_primary_val_table).to_frame("value"))

schema_summary, schema_detail = schema_alignment_report(
    fold1_train_policy_union,
    fold1_primary_val_table,
)

print("06.1 union_shared train / validation schema alignment")
display(schema_summary.to_frame("value"))
display(schema_detail)

structured_rmse_fold1_primary = rmse_from_series(
    fold1_primary_val_table["target_iv"],
    fold1_primary_val_table["structured_winner_pred"],
)

ridge_fold1_primary_summary, ridge_fold1_primary_preds, ridge_fold1_primary_model, ridge_fold1_primary_cols = evaluate_ridge_model(
    fold1_train_policy_union,
    fold1_primary_val_table,
    feature_cols=FULL_HISTGB_ABLATION_FEATURES,
    target_col="target_iv",
    alpha=1.0,
    prediction_mode="direct_iv",
)

pruned_histgb_fold1_primary_summary, pruned_histgb_fold1_primary_preds, pruned_histgb_fold1_primary_model = evaluate_histgb_model(
    fold1_train_policy_union,
    fold1_primary_val_table,
    feature_cols=PHASE6_PRUNED_HISTGB_FEATURES,
    target_col="target_residual",
    learning_rate=0.05,
    max_depth=3,
    max_iter=300,
    min_samples_leaf=10,
    prediction_mode="residual",
)

fold1_policy_panel = build_prediction_panel_for_policy_eval(
    fold1_primary_val_table,
    structured_pred_df=fold1_primary_val_table.rename(columns={"structured_winner_pred": "iv_pred"}),
    ridge_pred_df=ridge_fold1_primary_preds,
    pruned_histgb_pred_df=pruned_histgb_fold1_primary_preds,
)

hybrid_fold1_primary_preds = apply_hybrid_slice_no_hard_case_override_pruned(fold1_policy_panel).copy()
hybrid_fold1_primary_rmse = rmse_from_series(
    hybrid_fold1_primary_preds["target_iv"],
    hybrid_fold1_primary_preds["iv_pred"],
)

phase61_fold1_primary_sanity = pd.DataFrame(
    [
        {
            "model": "structured_winner",
            "val_rmse": structured_rmse_fold1_primary,
        },
        {
            "model": "ridge_direct_full",
            "val_rmse": ridge_fold1_primary_summary["val_rmse"],
        },
        {
            "model": "histgb_pruned_v1_no_anchor_no_regime",
            "val_rmse": pruned_histgb_fold1_primary_summary["val_rmse"],
        },
        {
            "model": "hybrid_slice_no_hard_case_override_pruned",
            "val_rmse": hybrid_fold1_primary_rmse,
        },
    ]
).sort_values("val_rmse")

print("06.1 Fold 1 / primary_realistic sanity reconstruction under union_shared")
display(phase61_fold1_primary_sanity)

phase61_fold1_hybrid_route_mix = (
    hybrid_fold1_primary_preds["hybrid_route"]
    .value_counts(dropna=False)
    .rename_axis("hybrid_route")
    .to_frame("count")
)

print("06.1 Fold 1 hybrid route mix sanity check")
display(phase61_fold1_hybrid_route_mix)


06.1 union_shared Fold 1 training table summary


,value
n_rows,310
n_dates,20
date_min,2025-03-24
date_max,2025-04-18
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,21.4990
mean_target_residual,0.2041
rmse_structured_winner,1.0367


06.1 Fold 1 primary_realistic validation table summary


,value
n_rows,39
n_dates,5
date_min,2025-04-21
date_max,2025-04-25
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,19.5990
mean_target_residual,0.3840
rmse_structured_winner,1.3592


06.1 union_shared train / validation schema alignment


...

,train_only_cols,val_only_cols
0,source_protocol_for_training_rows,NaN
1,training_policy,NaN


06.1 Fold 1 / primary_realistic sanity reconstruction under union_shared


,model,val_rmse
3,hybrid_slice_no_hard_case_override_pruned,0.8157
2,histgb_pruned_v1_no_anchor_no_regime,0.9232
1,ridge_direct_full,0.9428
0,structured_winner,1.3592


06.1 Fold 1 hybrid route mix sanity check


,count
hybrid_route,
structured_winner,19
histgb_pruned_v1_no_anchor_no_regime,17
ridge_direct_full,3


In [15]:
validation_table_cache = {}

for protocol_name in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        val_table = build_validation_table_for_fold_protocol(
            train,
            fold_row,
            protocol_name=protocol_name,
            lookback_dates=20,
        )

        validation_table_cache[(protocol_name, fold_id)] = val_table

print("06.1 validation table cache")
display(
    pd.DataFrame(
        [
            {
                "protocol": protocol_name,
                "fold": fold_id,
                "n_val_rows": len(val_table),
                "n_val_dates": val_table["target_date"].nunique(),
            }
            for (protocol_name, fold_id), val_table in validation_table_cache.items()
        ]
    ).sort_values(["protocol", "fold"]).reset_index(drop=True)
)


06.1 validation table cache


,protocol,fold,n_val_rows,n_val_dates
0,primary_realistic,1,39,5
1,primary_realistic,2,38,5
2,primary_realistic,3,39,5
3,primary_realistic,4,40,5
4,stress_test,1,39,5
5,stress_test,2,38,5
6,stress_test,3,39,5
7,stress_test,4,40,5


In [16]:
TRAINING_POLICIES = ["primary_only", "stress_only", "union_shared"]

training_table_cache = {}

for training_policy in TRAINING_POLICIES:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])

        train_table = build_training_table_for_policy_fold(
            train,
            fold_row,
            training_policy=training_policy,
            lookback_dates=20,
        )

        training_table_cache[(training_policy, fold_id)] = train_table

print("06.1 training table cache")
display(
    pd.DataFrame(
        [
            {
                "training_policy": training_policy,
                "fold": fold_id,
                "n_train_rows": len(train_table),
                "n_train_dates": train_table["target_date"].nunique(),
                "source_protocols_used": ", ".join(sorted(train_table["source_protocol_for_training_rows"].unique())),
            }
            for (training_policy, fold_id), train_table in training_table_cache.items()
        ]
    ).sort_values(["training_policy", "fold"]).reset_index(drop=True)
)


06.1 training table cache


,training_policy,fold,n_train_rows,n_train_dates,source_protocols_used
0,primary_only,1,155,20,primary_realistic
1,primary_only,2,155,20,primary_realistic
2,primary_only,3,155,20,primary_realistic
3,primary_only,4,156,20,primary_realistic
4,stress_only,1,155,20,stress_test
5,stress_only,2,155,20,stress_test
6,stress_only,3,155,20,stress_test
7,stress_only,4,156,20,stress_test
8,union_shared,1,310,20,"primary_realistic, stress_test"
9,union_shared,2,310,20,"primary_realistic, stress_test"


In [17]:
def run_ridge_direct_full_policy(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model, cols = evaluate_ridge_model(
        train_table,
        val_table,
        feature_cols=FULL_HISTGB_ABLATION_FEATURES,
        target_col="target_iv",
        alpha=1.0,
        prediction_mode="direct_iv",
    )
    summary = {"model": "ridge_direct_full", **summary}
    return summary, pred_df


def run_histgb_pruned_policy(train_table: pd.DataFrame, val_table: pd.DataFrame):
    summary, pred_df, model = evaluate_histgb_model(
        train_table,
        val_table,
        feature_cols=PHASE6_PRUNED_HISTGB_FEATURES,
        target_col="target_residual",
        learning_rate=0.05,
        max_depth=3,
        max_iter=300,
        min_samples_leaf=10,
        prediction_mode="residual",
    )
    summary = {"model": "histgb_pruned_v1_no_anchor_no_regime", **summary}
    return summary, pred_df


def run_hybrid_pruned_policy(
    val_table: pd.DataFrame,
    ridge_pred_df: pd.DataFrame,
    pruned_histgb_pred_df: pd.DataFrame,
):
    structured_pred_df = val_table[["row_id", "structured_winner_pred"]].rename(
        columns={"structured_winner_pred": "iv_pred"}
    )

    panel_df = build_prediction_panel_for_policy_eval(
        val_table,
        structured_pred_df=structured_pred_df,
        ridge_pred_df=ridge_pred_df,
        pruned_histgb_pred_df=pruned_histgb_pred_df,
    )

    pred_df = apply_hybrid_slice_no_hard_case_override_pruned(panel_df).copy()

    summary = {
        "model": "hybrid_slice_no_hard_case_override_pruned",
        "target_col": "target_iv",
        "prediction_mode": "mixed",
        "val_rmse": rmse_from_series(pred_df["target_iv"], pred_df["iv_pred"]),
        "val_mae": float((pred_df["target_iv"] - pred_df["iv_pred"]).abs().mean()),
    }

    return summary, pred_df


In [18]:
phase61_policy_rows = []
phase61_policy_prediction_cache = {}

for training_policy in TRAINING_POLICIES:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])
        train_table = training_table_cache[(training_policy, fold_id)]

        for eval_protocol in PROTOCOLS:
            val_table = validation_table_cache[(eval_protocol, fold_id)]

            ridge_summary, ridge_pred_df = run_ridge_direct_full_policy(train_table, val_table)
            pruned_histgb_summary, pruned_histgb_pred_df = run_histgb_pruned_policy(train_table, val_table)
            hybrid_summary, hybrid_pred_df = run_hybrid_pruned_policy(
                val_table,
                ridge_pred_df=ridge_pred_df,
                pruned_histgb_pred_df=pruned_histgb_pred_df,
            )

            for summary, pred_df, model_name in [
                (ridge_summary, ridge_pred_df, "ridge_direct_full"),
                (pruned_histgb_summary, pruned_histgb_pred_df, "histgb_pruned_v1_no_anchor_no_regime"),
                (hybrid_summary, hybrid_pred_df, "hybrid_slice_no_hard_case_override_pruned"),
            ]:
                phase61_policy_rows.append(
                    {
                        "training_policy": training_policy,
                        "eval_protocol": eval_protocol,
                        "fold": fold_id,
                        "model": model_name,
                        "n_train_rows": len(train_table),
                        "n_val_rows": len(val_table),
                        **summary,
                    }
                )

                pred_df = pred_df.copy()
                pred_df["training_policy"] = training_policy
                pred_df["eval_protocol"] = eval_protocol
                pred_df["fold"] = fold_id
                pred_df["model"] = model_name
                phase61_policy_prediction_cache[(training_policy, eval_protocol, fold_id, model_name)] = pred_df

phase61_policy_results = (
    pd.DataFrame(phase61_policy_rows)
    .sort_values(["training_policy", "eval_protocol", "fold", "val_rmse", "model"])
    .reset_index(drop=True)
)

print("06.1 training-policy comparison results")
display(phase61_policy_results)


06.1 training-policy comparison results


,training_policy,eval_protocol,fold,model,n_train_rows,n_val_rows,alpha,target_col,prediction_mode,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse,learning_rate,max_depth,max_iter,min_samples_leaf,val_mae
0,primary_only,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,155,39,NaN,target_iv,mixed,NaN,NaN,0.7600,NaN,NaN,NaN,NaN,NaN,0.6206
1,primary_only,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,155,39,NaN,target_residual,residual,51.0000,0.0926,0.8428,1.3592,0.0500,3.0000,300.0000,10.0000,NaN
2,primary_only,primary_realistic,1,ridge_direct_full,155,39,1.0000,target_iv,direct_iv,65.0000,21.3784,1.0556,1.3592,NaN,NaN,NaN,NaN,NaN
3,primary_only,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,155,38,NaN,target_iv,mixed,NaN,NaN,0.9063,NaN,NaN,NaN,NaN,NaN,0.7201
4,primary_only,primary_realistic,2,ridge_direct_full,155,38,1.0000,target_iv,direct_iv,65.0000,20.7935,0.9294,1.0373,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,union_shared,stress_test,3,ridge_direct_full,310,39,1.0000,target_iv,direct_iv,65.0000,22.0979,0.8005,1.0614,NaN,NaN,NaN,NaN,NaN
68,union_shared,stress_test,3,histgb_pruned_v1_no_anchor_no_regime,310,39,NaN,target_residual,residual,51.0000,0.1646,0.9091,1.0614,0.0500,3.0000,300.0000,10.0000,NaN
69,union_shared,stress_test,4,ridge_direct_full,312,40,1.0000,target_iv,direct_iv,65.0000,21.5677,0.7018,1.1892,NaN,NaN,NaN,NaN,NaN
70,union_shared,stress_test,4,hybrid_slice_no_hard_case_override_pruned,312,40,NaN,target_iv,mixed,NaN,NaN,0.8060,NaN,NaN,NaN,NaN,NaN,0.5879


In [19]:
phase61_policy_protocol_summary = (
    phase61_policy_results
    .groupby(["training_policy", "eval_protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["training_policy", "eval_protocol", "mean_rmse"])
)

print("06.1 policy protocol summary")
display(phase61_policy_protocol_summary)


06.1 policy protocol summary


,training_policy,eval_protocol,model,mean_rmse,min_rmse,max_rmse
1,primary_only,primary_realistic,hybrid_slice_no_hard_case_override_pruned,0.8158,0.7600,0.9063
0,primary_only,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8130,0.9512
2,primary_only,primary_realistic,ridge_direct_full,0.9245,0.8397,1.0556
4,primary_only,stress_test,hybrid_slice_no_hard_case_override_pruned,0.8148,0.7342,0.9753
5,primary_only,stress_test,ridge_direct_full,0.9239,0.8710,1.0281
3,primary_only,stress_test,histgb_pruned_v1_no_anchor_no_regime,0.9922,0.9060,1.1164
7,stress_only,primary_realistic,hybrid_slice_no_hard_case_override_pruned,0.8264,0.7795,0.9162
8,stress_only,primary_realistic,ridge_direct_full,0.8546,0.8330,0.8853
6,stress_only,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,0.9108,0.8372,1.0329
10,stress_only,stress_test,hybrid_slice_no_hard_case_override_pruned,0.7917,0.7127,0.8407


In [20]:
hybrid_policy_summary = phase61_policy_protocol_summary.loc[
    phase61_policy_protocol_summary["model"] == "hybrid_slice_no_hard_case_override_pruned"
].copy()

hybrid_policy_pivot = (
    hybrid_policy_summary
    .pivot(index="training_policy", columns="eval_protocol", values="mean_rmse")
    .reset_index()
)

hybrid_policy_pivot["avg_two_protocols"] = (
    hybrid_policy_pivot["primary_realistic"] + hybrid_policy_pivot["stress_test"]
) / 2.0

hybrid_policy_pivot["max_two_protocols"] = hybrid_policy_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("06.1 hybrid training-policy decision pivot")
display(hybrid_policy_pivot.sort_values(["primary_realistic", "max_two_protocols", "avg_two_protocols"]))


06.1 hybrid training-policy decision pivot


eval_protocol,training_policy,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,primary_only,0.8158,0.8148,0.8153,0.8158
1,stress_only,0.8264,0.7917,0.8090,0.8264
2,union_shared,0.8397,0.7990,0.8193,0.8397


In [21]:
phase61_all_model_policy_pivot = (
    phase61_policy_protocol_summary
    .pivot(index=["training_policy", "model"], columns="eval_protocol", values="mean_rmse")
    .reset_index()
)

phase61_all_model_policy_pivot["avg_two_protocols"] = (
    phase61_all_model_policy_pivot["primary_realistic"] + phase61_all_model_policy_pivot["stress_test"]
) / 2.0

phase61_all_model_policy_pivot["max_two_protocols"] = phase61_all_model_policy_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("06.1 all-model training-policy pivot")
display(
    phase61_all_model_policy_pivot.sort_values(
        ["model", "primary_realistic", "max_two_protocols", "avg_two_protocols"]
    )
)


06.1 all-model training-policy pivot


eval_protocol,training_policy,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,primary_only,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.9922,0.9324,0.9922
6,union_shared,histgb_pruned_v1_no_anchor_no_regime,0.9006,0.9047,0.9027,0.9047
3,stress_only,histgb_pruned_v1_no_anchor_no_regime,0.9108,0.8989,0.9049,0.9108
1,primary_only,hybrid_slice_no_hard_case_override_pruned,0.8158,0.8148,0.8153,0.8158
4,stress_only,hybrid_slice_no_hard_case_override_pruned,0.8264,0.7917,0.8090,0.8264
7,union_shared,hybrid_slice_no_hard_case_override_pruned,0.8397,0.7990,0.8193,0.8397
8,union_shared,ridge_direct_full,0.8403,0.8069,0.8236,0.8403
5,stress_only,ridge_direct_full,0.8546,0.8463,0.8505,0.8546
2,primary_only,ridge_direct_full,0.9245,0.9239,0.9242,0.9245


In [22]:
phase61_hybrid_policy_fold_view = phase61_policy_results.loc[
    phase61_policy_results["model"] == "hybrid_slice_no_hard_case_override_pruned",
    ["training_policy", "eval_protocol", "fold", "val_rmse"]
].sort_values(["training_policy", "eval_protocol", "fold"])

print("06.1 hybrid policy fold view")
display(phase61_hybrid_policy_fold_view)


06.1 hybrid policy fold view


,training_policy,eval_protocol,fold,val_rmse
0,primary_only,primary_realistic,1,0.7600
3,primary_only,primary_realistic,2,0.9063
6,primary_only,primary_realistic,3,0.7866
9,primary_only,primary_realistic,4,0.8102
12,primary_only,stress_test,1,0.8075
15,primary_only,stress_test,2,0.7424
18,primary_only,stress_test,3,0.7342
22,primary_only,stress_test,4,0.9753
24,stress_only,primary_realistic,1,0.7795
28,stress_only,primary_realistic,2,0.9162


## Shared training policy locked

Based on the policy-comparison results, the shared ML training policy for this notebook is now locked as:

- `primary_only`

Reason:
- it gives the best `primary_realistic` performance for the leading hybrid
- it also gives the best `max_two_protocols`
- `union_shared` is clearly worse
- `stress_only` improves the conservative protocol but gives back too much on the primary ranking metric

The remaining work in this notebook now uses:

- one shared policy: `primary_only`
- narrow finalist-only tuning
- fixed hybrid routing logic


In [29]:
LOCKED_SHARED_POLICY = "primary_only"

locked_train_table_cache = {
    fold_id: training_table_cache[(LOCKED_SHARED_POLICY, fold_id)].copy()
    for fold_id in fold_plan["fold"].astype(int).tolist()
}

print("Locked shared training policy:", LOCKED_SHARED_POLICY)
display(
    pd.DataFrame(
        [
            {
                "fold": fold_id,
                "n_train_rows": len(train_table),
                "n_train_dates": train_table["target_date"].nunique(),
                "source_protocols_used": ", ".join(sorted(train_table["source_protocol_for_training_rows"].unique())),
            }
            for fold_id, train_table in locked_train_table_cache.items()
        ]
    ).sort_values("fold").reset_index(drop=True)
)


Locked shared training policy: primary_only


,fold,n_train_rows,n_train_dates,source_protocols_used
0,1,155,20,primary_realistic
1,2,155,20,primary_realistic
2,3,155,20,primary_realistic
3,4,156,20,primary_realistic


In [30]:
RIDGE_TUNING_GRID = pd.DataFrame(
    [
        {"config_id": "ridge_a_0p1", "alpha": 0.1},
        {"config_id": "ridge_a_0p3", "alpha": 0.3},
        {"config_id": "ridge_a_1p0", "alpha": 1.0},
        {"config_id": "ridge_a_3p0", "alpha": 3.0},
        {"config_id": "ridge_a_10p0", "alpha": 10.0},
    ]
)

HISTGB_TUNING_GRID = pd.DataFrame(
    [
        {"config_id": "hgb_base", "learning_rate": 0.05, "max_depth": 3, "max_iter": 300, "min_samples_leaf": 10},
        {"config_id": "hgb_lr_0p03_iter_500", "learning_rate": 0.03, "max_depth": 3, "max_iter": 500, "min_samples_leaf": 10},
        {"config_id": "hgb_depth_2", "learning_rate": 0.05, "max_depth": 2, "max_iter": 300, "min_samples_leaf": 10},
        {"config_id": "hgb_depth_4", "learning_rate": 0.05, "max_depth": 4, "max_iter": 300, "min_samples_leaf": 10},
        {"config_id": "hgb_leaf_5", "learning_rate": 0.05, "max_depth": 3, "max_iter": 300, "min_samples_leaf": 5},
        {"config_id": "hgb_lr_0p08", "learning_rate": 0.08, "max_depth": 3, "max_iter": 300, "min_samples_leaf": 10},
    ]
)

print("Ridge tuning grid")
display(RIDGE_TUNING_GRID)

print("HistGB tuning grid")
display(HISTGB_TUNING_GRID)


Ridge tuning grid


,config_id,alpha
0,ridge_a_0p1,0.1000
1,ridge_a_0p3,0.3000
2,ridge_a_1p0,1.0000
3,ridge_a_3p0,3.0000
4,ridge_a_10p0,10.0000


HistGB tuning grid


,config_id,learning_rate,max_depth,max_iter,min_samples_leaf
0,hgb_base,0.0500,3,300,10
1,hgb_lr_0p03_iter_500,0.0300,3,500,10
2,hgb_depth_2,0.0500,2,300,10
3,hgb_depth_4,0.0500,4,300,10
4,hgb_leaf_5,0.0500,3,300,5
5,hgb_lr_0p08,0.0800,3,300,10


In [31]:
ridge_tuning_rows = []
ridge_tuning_prediction_cache = {}

for _, cfg in RIDGE_TUNING_GRID.iterrows():
    config_id = cfg["config_id"]
    alpha = float(cfg["alpha"])

    for eval_protocol in PROTOCOLS:
        for _, fold_row in fold_plan.iterrows():
            fold_id = int(fold_row["fold"])

            train_table = locked_train_table_cache[fold_id]
            val_table = validation_table_cache[(eval_protocol, fold_id)]

            summary, pred_df, model, cols = evaluate_ridge_model(
                train_table,
                val_table,
                feature_cols=FULL_HISTGB_ABLATION_FEATURES,
                target_col="target_iv",
                alpha=alpha,
                prediction_mode="direct_iv",
            )

            ridge_tuning_rows.append(
                {
                    "config_id": config_id,
                    "alpha": alpha,
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "model": "ridge_direct_full",
                    **summary,
                }
            )

            pred_df = pred_df.copy()
            pred_df["config_id"] = config_id
            pred_df["eval_protocol"] = eval_protocol
            pred_df["fold"] = fold_id
            ridge_tuning_prediction_cache[(config_id, eval_protocol, fold_id)] = pred_df

ridge_tuning_results = (
    pd.DataFrame(ridge_tuning_rows)
    .sort_values(["config_id", "eval_protocol", "fold"])
    .reset_index(drop=True)
)

print("06.1 Ridge tuning results")
display(ridge_tuning_results)


06.1 Ridge tuning results


,config_id,alpha,eval_protocol,fold,model,target_col,prediction_mode,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
0,ridge_a_0p1,0.1000,primary_realistic,1,ridge_direct_full,target_iv,direct_iv,65,21.3784,1.0551,1.3592
1,ridge_a_0p1,0.1000,primary_realistic,2,ridge_direct_full,target_iv,direct_iv,65,20.7935,0.9378,1.0373
2,ridge_a_0p1,0.1000,primary_realistic,3,ridge_direct_full,target_iv,direct_iv,65,21.9630,0.9017,0.7840
3,ridge_a_0p1,0.1000,primary_realistic,4,ridge_direct_full,target_iv,direct_iv,65,21.4115,0.8684,1.0748
4,ridge_a_0p1,0.1000,stress_test,1,ridge_direct_full,target_iv,direct_iv,65,21.3784,1.0285,1.3163
5,ridge_a_0p1,0.1000,stress_test,2,ridge_direct_full,target_iv,direct_iv,65,20.7935,0.8957,1.1719
6,ridge_a_0p1,0.1000,stress_test,3,ridge_direct_full,target_iv,direct_iv,65,21.9630,0.8990,1.0614
7,ridge_a_0p1,0.1000,stress_test,4,ridge_direct_full,target_iv,direct_iv,65,21.4115,0.8879,1.1892
8,ridge_a_0p3,0.3000,primary_realistic,1,ridge_direct_full,target_iv,direct_iv,65,21.3784,1.0627,1.3592
9,ridge_a_0p3,0.3000,primary_realistic,2,ridge_direct_full,target_iv,direct_iv,65,20.7935,0.9398,1.0373


In [32]:
histgb_tuning_rows = []
histgb_tuning_prediction_cache = {}

for _, cfg in HISTGB_TUNING_GRID.iterrows():
    config_id = cfg["config_id"]

    for eval_protocol in PROTOCOLS:
        for _, fold_row in fold_plan.iterrows():
            fold_id = int(fold_row["fold"])

            train_table = locked_train_table_cache[fold_id]
            val_table = validation_table_cache[(eval_protocol, fold_id)]

            summary, pred_df, model = evaluate_histgb_model(
                train_table,
                val_table,
                feature_cols=PHASE6_PRUNED_HISTGB_FEATURES,
                target_col="target_residual",
                learning_rate=float(cfg["learning_rate"]),
                max_depth=int(cfg["max_depth"]),
                max_iter=int(cfg["max_iter"]),
                min_samples_leaf=int(cfg["min_samples_leaf"]),
                prediction_mode="residual",
            )

            histgb_tuning_rows.append(
                {
                    "config_id": config_id,
                    "learning_rate": float(cfg["learning_rate"]),
                    "max_depth": int(cfg["max_depth"]),
                    "max_iter": int(cfg["max_iter"]),
                    "min_samples_leaf": int(cfg["min_samples_leaf"]),
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "model": "histgb_pruned_v1_no_anchor_no_regime",
                    **summary,
                }
            )

            pred_df = pred_df.copy()
            pred_df["config_id"] = config_id
            pred_df["eval_protocol"] = eval_protocol
            pred_df["fold"] = fold_id
            histgb_tuning_prediction_cache[(config_id, eval_protocol, fold_id)] = pred_df

histgb_tuning_results = (
    pd.DataFrame(histgb_tuning_rows)
    .sort_values(["config_id", "eval_protocol", "fold"])
    .reset_index(drop=True)
)

print("06.1 HistGB tuning results")
display(histgb_tuning_results)


06.1 HistGB tuning results


,config_id,learning_rate,max_depth,max_iter,min_samples_leaf,eval_protocol,fold,model,target_col,prediction_mode,n_features_after_encoding,train_target_mean,val_rmse,structured_winner_rmse
0,hgb_base,0.0500,3,300,10,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.0926,0.8428,1.3592
1,hgb_base,0.0500,3,300,10,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.1957,0.9512,1.0373
2,hgb_base,0.0500,3,300,10,primary_realistic,3,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.1308,0.8130,0.7840
3,hgb_base,0.0500,3,300,10,primary_realistic,4,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.2026,0.8832,1.0748
4,hgb_base,0.0500,3,300,10,stress_test,1,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.0926,0.9060,1.3163
5,hgb_base,0.0500,3,300,10,stress_test,2,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.1957,0.9559,1.1719
6,hgb_base,0.0500,3,300,10,stress_test,3,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.1308,0.9906,1.0614
7,hgb_base,0.0500,3,300,10,stress_test,4,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.2026,1.1164,1.1892
8,hgb_depth_2,0.0500,2,300,10,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.0926,0.9051,1.3592
9,hgb_depth_2,0.0500,2,300,10,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,target_residual,residual,51,0.1957,0.9327,1.0373


In [33]:
ridge_tuning_summary = (
    ridge_tuning_results
    .groupby(["config_id", "alpha", "eval_protocol"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["config_id", "eval_protocol"])
)

print("06.1 Ridge tuning summary")
display(ridge_tuning_summary)


06.1 Ridge tuning summary


,config_id,alpha,eval_protocol,mean_rmse,min_rmse,max_rmse
0,ridge_a_0p1,0.1000,primary_realistic,0.9408,0.8684,1.0551
1,ridge_a_0p1,0.1000,stress_test,0.9278,0.8879,1.0285
2,ridge_a_0p3,0.3000,primary_realistic,0.9370,0.8576,1.0627
3,ridge_a_0p3,0.3000,stress_test,0.9274,0.8822,1.0263
4,ridge_a_10p0,10.0000,primary_realistic,0.8747,0.7869,1.0042
5,ridge_a_10p0,10.0000,stress_test,0.8898,0.8246,1.0231
6,ridge_a_1p0,1.0000,primary_realistic,0.9245,0.8397,1.0556
7,ridge_a_1p0,1.0000,stress_test,0.9239,0.8710,1.0281
8,ridge_a_3p0,3.0000,primary_realistic,0.9040,0.8159,1.0363
9,ridge_a_3p0,3.0000,stress_test,0.9129,0.8522,1.0291


In [37]:
ridge_tuning_pivot = (
    ridge_tuning_summary
    .pivot(index=["config_id", "alpha"], columns="eval_protocol", values="mean_rmse")
    .reset_index()
)

ridge_tuning_pivot["avg_two_protocols"] = (
    ridge_tuning_pivot["primary_realistic"] + ridge_tuning_pivot["stress_test"]
) / 2.0

ridge_tuning_pivot["max_two_protocols"] = ridge_tuning_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("06.1 Ridge tuning decision pivot")
display(ridge_tuning_pivot.sort_values(["primary_realistic", "max_two_protocols", "avg_two_protocols"]))


06.1 Ridge tuning decision pivot


eval_protocol,config_id,alpha,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
2,ridge_a_10p0,10.0000,0.8747,0.8898,0.8823,0.8898
4,ridge_a_3p0,3.0000,0.9040,0.9129,0.9084,0.9129
3,ridge_a_1p0,1.0000,0.9245,0.9239,0.9242,0.9245
1,ridge_a_0p3,0.3000,0.9370,0.9274,0.9322,0.9370
0,ridge_a_0p1,0.1000,0.9408,0.9278,0.9343,0.9408


In [38]:
histgb_tuning_summary = (
    histgb_tuning_results
    .groupby(["config_id", "learning_rate", "max_depth", "max_iter", "min_samples_leaf", "eval_protocol"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["config_id", "eval_protocol"])
)

print("06.1 HistGB tuning summary")
display(histgb_tuning_summary)


06.1 HistGB tuning summary


,config_id,learning_rate,max_depth,max_iter,min_samples_leaf,eval_protocol,mean_rmse,min_rmse,max_rmse
0,hgb_base,0.0500,3,300,10,primary_realistic,0.8725,0.8130,0.9512
1,hgb_base,0.0500,3,300,10,stress_test,0.9922,0.9060,1.1164
2,hgb_depth_2,0.0500,2,300,10,primary_realistic,0.8932,0.8414,0.9327
3,hgb_depth_2,0.0500,2,300,10,stress_test,0.9799,0.8965,1.0915
4,hgb_depth_4,0.0500,4,300,10,primary_realistic,0.8734,0.8180,0.9554
5,hgb_depth_4,0.0500,4,300,10,stress_test,1.0446,0.9502,1.1805
6,hgb_leaf_5,0.0500,3,300,5,primary_realistic,0.8730,0.8197,0.9760
7,hgb_leaf_5,0.0500,3,300,5,stress_test,1.0866,1.0497,1.1221
8,hgb_lr_0p03_iter_500,0.0300,3,500,10,primary_realistic,0.8761,0.8169,0.9458
9,hgb_lr_0p03_iter_500,0.0300,3,500,10,stress_test,1.0048,0.9234,1.1352


In [39]:
histgb_tuning_pivot = (
    histgb_tuning_summary
    .pivot(
        index=["config_id", "learning_rate", "max_depth", "max_iter", "min_samples_leaf"],
        columns="eval_protocol",
        values="mean_rmse",
    )
    .reset_index()
)

histgb_tuning_pivot["avg_two_protocols"] = (
    histgb_tuning_pivot["primary_realistic"] + histgb_tuning_pivot["stress_test"]
) / 2.0

histgb_tuning_pivot["max_two_protocols"] = histgb_tuning_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("06.1 HistGB tuning decision pivot")
display(histgb_tuning_pivot.sort_values(["primary_realistic", "max_two_protocols", "avg_two_protocols"]))


06.1 HistGB tuning decision pivot


eval_protocol,config_id,learning_rate,max_depth,max_iter,min_samples_leaf,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,hgb_base,0.0500,3,300,10,0.8725,0.9922,0.9324,0.9922
3,hgb_leaf_5,0.0500,3,300,5,0.8730,1.0866,0.9798,1.0866
2,hgb_depth_4,0.0500,4,300,10,0.8734,1.0446,0.9590,1.0446
4,hgb_lr_0p03_iter_500,0.0300,3,500,10,0.8761,1.0048,0.9405,1.0048
5,hgb_lr_0p08,0.0800,3,300,10,0.8919,1.0349,0.9634,1.0349
1,hgb_depth_2,0.0500,2,300,10,0.8932,0.9799,0.9366,0.9799


In [40]:
best_ridge_config = ridge_tuning_pivot.sort_values(
    ["primary_realistic", "max_two_protocols", "avg_two_protocols"]
).reset_index(drop=True).iloc[0]

best_histgb_config = histgb_tuning_pivot.sort_values(
    ["primary_realistic", "max_two_protocols", "avg_two_protocols"]
).reset_index(drop=True).iloc[0]

print("Provisional best Ridge config")
display(pd.DataFrame([best_ridge_config]))

print("Provisional best pruned HistGB config")
display(pd.DataFrame([best_histgb_config]))


Provisional best Ridge config


eval_protocol,config_id,alpha,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,ridge_a_10p0,10.0000,0.8747,0.8898,0.8823,0.8898


Provisional best pruned HistGB config


eval_protocol,config_id,learning_rate,max_depth,max_iter,min_samples_leaf,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
0,hgb_base,0.0500,3,300,10,0.8725,0.9922,0.9324,0.9922


In [41]:
LOCKED_RIDGE_CONFIG_ID = "ridge_a_10p0"
LOCKED_HISTGB_CONFIG_ID = "hgb_base"

print("Locked shared policy:", LOCKED_SHARED_POLICY)
print("Locked Ridge config:", LOCKED_RIDGE_CONFIG_ID)
print("Locked HistGB config:", LOCKED_HISTGB_CONFIG_ID)


Locked shared policy: primary_only
Locked Ridge config: ridge_a_10p0
Locked HistGB config: hgb_base


In [42]:
phase61_tuned_rows = []
phase61_tuned_prediction_cache = {}

for eval_protocol in PROTOCOLS:
    for _, fold_row in fold_plan.iterrows():
        fold_id = int(fold_row["fold"])
        val_table = validation_table_cache[(eval_protocol, fold_id)]

        structured_pred_df = val_table[["row_id", "target_iv", "structured_winner_pred"]].copy()
        structured_pred_df["iv_pred"] = structured_pred_df["structured_winner_pred"]

        ridge_pred_df = ridge_tuning_prediction_cache[(LOCKED_RIDGE_CONFIG_ID, eval_protocol, fold_id)].copy()
        histgb_pred_df = histgb_tuning_prediction_cache[(LOCKED_HISTGB_CONFIG_ID, eval_protocol, fold_id)].copy()

        panel_df = build_prediction_panel_for_policy_eval(
            val_table,
            structured_pred_df=structured_pred_df[["row_id", "iv_pred"]],
            ridge_pred_df=ridge_pred_df,
            pruned_histgb_pred_df=histgb_pred_df,
        )

        hybrid_pred_df = apply_hybrid_slice_no_hard_case_override_pruned(panel_df).copy()

        structured_rmse = rmse_from_series(val_table["target_iv"], structured_pred_df["iv_pred"])
        ridge_rmse = rmse_from_series(val_table["target_iv"], ridge_pred_df["iv_pred"])
        histgb_rmse = rmse_from_series(val_table["target_iv"], histgb_pred_df["iv_pred"])
        hybrid_rmse = rmse_from_series(val_table["target_iv"], hybrid_pred_df["iv_pred"])

        phase61_tuned_rows.extend(
            [
                {
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "model": "structured_winner",
                    "val_rmse": structured_rmse,
                    "training_policy": LOCKED_SHARED_POLICY,
                    "component_config": "(none)",
                },
                {
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "model": "ridge_direct_full",
                    "val_rmse": ridge_rmse,
                    "training_policy": LOCKED_SHARED_POLICY,
                    "component_config": LOCKED_RIDGE_CONFIG_ID,
                },
                {
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "model": "histgb_pruned_v1_no_anchor_no_regime",
                    "val_rmse": histgb_rmse,
                    "training_policy": LOCKED_SHARED_POLICY,
                    "component_config": LOCKED_HISTGB_CONFIG_ID,
                },
                {
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "model": "hybrid_slice_no_hard_case_override_pruned",
                    "val_rmse": hybrid_rmse,
                    "training_policy": LOCKED_SHARED_POLICY,
                    "component_config": f"ridge={LOCKED_RIDGE_CONFIG_ID}; histgb={LOCKED_HISTGB_CONFIG_ID}",
                },
            ]
        )

        phase61_tuned_prediction_cache[(eval_protocol, fold_id, "structured_winner")] = structured_pred_df.copy()
        phase61_tuned_prediction_cache[(eval_protocol, fold_id, "ridge_direct_full")] = ridge_pred_df.copy()
        phase61_tuned_prediction_cache[(eval_protocol, fold_id, "histgb_pruned_v1_no_anchor_no_regime")] = histgb_pred_df.copy()
        phase61_tuned_prediction_cache[(eval_protocol, fold_id, "hybrid_slice_no_hard_case_override_pruned")] = hybrid_pred_df.copy()

phase61_tuned_results = (
    pd.DataFrame(phase61_tuned_rows)
    .sort_values(["eval_protocol", "fold", "val_rmse", "model"])
    .reset_index(drop=True)
)

print("06.1 tuned finalist fold-level results")
display(phase61_tuned_results)


06.1 tuned finalist fold-level results


,eval_protocol,fold,model,val_rmse,training_policy,component_config
0,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.7558,primary_only,ridge=ridge_a_10p0; histgb=hgb_base
1,primary_realistic,1,histgb_pruned_v1_no_anchor_no_regime,0.8428,primary_only,hgb_base
2,primary_realistic,1,ridge_direct_full,1.0042,primary_only,ridge_a_10p0
3,primary_realistic,1,structured_winner,1.3592,primary_only,(none)
4,primary_realistic,2,ridge_direct_full,0.8717,primary_only,ridge_a_10p0
5,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,0.9007,primary_only,ridge=ridge_a_10p0; histgb=hgb_base
6,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,0.9512,primary_only,hgb_base
7,primary_realistic,2,structured_winner,1.0373,primary_only,(none)
8,primary_realistic,3,hybrid_slice_no_hard_case_override_pruned,0.7839,primary_only,ridge=ridge_a_10p0; histgb=hgb_base
9,primary_realistic,3,structured_winner,0.7840,primary_only,(none)


In [43]:
phase61_tuned_protocol_summary = (
    phase61_tuned_results
    .groupby(["eval_protocol", "model"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
    .sort_values(["eval_protocol", "mean_rmse"])
)

print("06.1 tuned finalist protocol summary")
display(phase61_tuned_protocol_summary)


06.1 tuned finalist protocol summary


,eval_protocol,model,mean_rmse,min_rmse,max_rmse
1,primary_realistic,hybrid_slice_no_hard_case_override_pruned,0.8125,0.7558,0.9007
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8130,0.9512
2,primary_realistic,ridge_direct_full,0.8747,0.7869,1.0042
3,primary_realistic,structured_winner,1.0638,0.7840,1.3592
5,stress_test,hybrid_slice_no_hard_case_override_pruned,0.8154,0.7370,0.9787
6,stress_test,ridge_direct_full,0.8898,0.8246,1.0231
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,0.9922,0.9060,1.1164
7,stress_test,structured_winner,1.1847,1.0614,1.3163


In [44]:
phase61_tuned_pivot = (
    phase61_tuned_protocol_summary
    .pivot(index="model", columns="eval_protocol", values="mean_rmse")
    .reset_index()
)

phase61_tuned_pivot["avg_two_protocols"] = (
    phase61_tuned_pivot["primary_realistic"] + phase61_tuned_pivot["stress_test"]
) / 2.0

phase61_tuned_pivot["max_two_protocols"] = phase61_tuned_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("06.1 tuned finalist decision pivot")
display(phase61_tuned_pivot.sort_values(["primary_realistic", "max_two_protocols", "avg_two_protocols"]))


06.1 tuned finalist decision pivot


eval_protocol,model,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
1,hybrid_slice_no_hard_case_override_pruned,0.8125,0.8154,0.8139,0.8154
0,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.9922,0.9324,0.9922
2,ridge_direct_full,0.8747,0.8898,0.8823,0.8898
3,structured_winner,1.0638,1.1847,1.1243,1.1847


In [49]:
untuned_ref = phase61_policy_protocol_summary.loc[
    phase61_policy_protocol_summary["training_policy"] == LOCKED_SHARED_POLICY,
    ["eval_protocol", "model", "mean_rmse"]
].rename(columns={"mean_rmse": "untuned_mean_rmse"})

phase61_tuned_vs_untuned = (
    phase61_tuned_protocol_summary
    .merge(untuned_ref, on=["eval_protocol", "model"], how="left")
    .assign(delta_tuned_minus_untuned=lambda df: df["mean_rmse"] - df["untuned_mean_rmse"])
    .sort_values(["eval_protocol", "mean_rmse"])
)

print("06.1 tuned vs untuned comparison")
display(phase61_tuned_vs_untuned)


06.1 tuned vs untuned comparison


,eval_protocol,model,mean_rmse,min_rmse,max_rmse,untuned_mean_rmse,delta_tuned_minus_untuned
0,primary_realistic,hybrid_slice_no_hard_case_override_pruned,0.8125,0.7558,0.9007,0.8158,-0.0033
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,0.8725,0.8130,0.9512,0.8725,0.0000
2,primary_realistic,ridge_direct_full,0.8747,0.7869,1.0042,0.9245,-0.0498
3,primary_realistic,structured_winner,1.0638,0.7840,1.3592,NaN,NaN
4,stress_test,hybrid_slice_no_hard_case_override_pruned,0.8154,0.7370,0.9787,0.8148,0.0005
5,stress_test,ridge_direct_full,0.8898,0.8246,1.0231,0.9239,-0.0341
6,stress_test,histgb_pruned_v1_no_anchor_no_regime,0.9922,0.9060,1.1164,0.9922,0.0000
7,stress_test,structured_winner,1.1847,1.0614,1.3163,NaN,NaN


In [47]:
phase61_tuned_hybrid_route_mix = []

for eval_protocol in PROTOCOLS:
    for fold_id in fold_plan["fold"].astype(int).tolist():
        pred_df = phase61_tuned_prediction_cache[(eval_protocol, fold_id, "hybrid_slice_no_hard_case_override_pruned")].copy()

        route_mix = (
            pred_df["hybrid_route"]
            .value_counts(dropna=False)
            .rename_axis("hybrid_route")
            .reset_index(name="count")
        )
        route_mix["eval_protocol"] = eval_protocol
        route_mix["fold"] = fold_id
        phase61_tuned_hybrid_route_mix.append(route_mix)

phase61_tuned_hybrid_route_mix = pd.concat(phase61_tuned_hybrid_route_mix, ignore_index=True)

print("06.1 tuned hybrid route mix")
display(
    phase61_tuned_hybrid_route_mix
    .groupby(["eval_protocol", "hybrid_route"], as_index=False)["count"]
    .sum()
    .sort_values(["eval_protocol", "count"], ascending=[True, False])
)


06.1 tuned hybrid route mix


,eval_protocol,hybrid_route,count
2,primary_realistic,structured_winner,88
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,58
1,primary_realistic,ridge_direct_full,10
5,stress_test,structured_winner,74
3,stress_test,histgb_pruned_v1_no_anchor_no_regime,69
4,stress_test,ridge_direct_full,13


In [48]:
phase61_tuned_fold_winners = (
    phase61_tuned_results
    .sort_values(["eval_protocol", "fold", "val_rmse"])
    .groupby(["eval_protocol", "fold"], as_index=False)
    .first()[["eval_protocol", "fold", "model", "val_rmse"]]
    .rename(columns={"model": "fold_winner", "val_rmse": "winning_rmse"})
)

print("06.1 tuned fold winners")
display(phase61_tuned_fold_winners)

print("06.1 tuned fold-win counts")
display(
    phase61_tuned_fold_winners["fold_winner"]
    .value_counts()
    .rename_axis("fold_winner")
    .to_frame("count")
)


06.1 tuned fold winners


,eval_protocol,fold,fold_winner,winning_rmse
0,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.7558
1,primary_realistic,2,ridge_direct_full,0.8717
2,primary_realistic,3,hybrid_slice_no_hard_case_override_pruned,0.7839
3,primary_realistic,4,ridge_direct_full,0.7869
4,stress_test,1,hybrid_slice_no_hard_case_override_pruned,0.8061
5,stress_test,2,hybrid_slice_no_hard_case_override_pruned,0.7370
6,stress_test,3,hybrid_slice_no_hard_case_override_pruned,0.7398
7,stress_test,4,ridge_direct_full,0.8246


06.1 tuned fold-win counts


,count
fold_winner,
hybrid_slice_no_hard_case_override_pruned,5
ridge_direct_full,3


## Phase 6.1 final training policy and narrow tuning: summary

This notebook took the confirmed finalists from `06_0_final_candidate_confirmation.ipynb` and performed the two remaining tasks before final inference:

1. choose one shared ML training policy
2. run narrow finalist-only tuning under that policy

The active models entering this notebook were:

- `hybrid_slice_no_hard_case_override_pruned`
- `histgb_pruned_v1_no_anchor_no_regime`
- `ridge_direct_full`

with:

- `structured_winner`

retained as an untuned benchmark.

---

## 1. Shared training policy comparison

This notebook compared three shared ML training policies:

- `primary_only`
- `stress_only`
- `union_shared`

The main selection rule was:

- rank primarily by mean RMSE on `primary_realistic`
- require acceptable confirmation on `stress_test`
- use lower `max_two_protocols` as the next tie-break
- prefer simpler choices when effectively tied

### Result

The shared policy selected in this notebook is:

- `primary_only`

Reason:
- it gave the best `primary_realistic` performance for the leading hybrid
- it also produced the best `max_two_protocols`
- `stress_only` improved the conservative protocol but gave back too much on the primary ranking metric
- `union_shared` was clearly weaker than both

So the final tuning work in this notebook was done under:

- `LOCKED_SHARED_POLICY = "primary_only"`

---

## 2. Narrow finalist-only tuning

Only narrow branch-level tuning was performed.

### Ridge
The tuning grid only varied:
- `alpha`

Selected Ridge configuration:
- `alpha = 10.0`

This improved Ridge materially relative to the untuned `alpha = 1.0` setup.

### Pruned HistGB
The tuning grid only varied:
- `learning_rate`
- `max_depth`
- `max_iter`
- `min_samples_leaf`

Result:
- the existing baseline pruned HistGB configuration remained the best

Selected pruned HistGB configuration:
- `learning_rate = 0.05`
- `max_depth = 3`
- `max_iter = 300`
- `min_samples_leaf = 10`

So HistGB did not need further tuning beyond the already selected baseline setup.

---

## 3. Rebuilt tuned finalist set

After policy selection and branch tuning, the finalists were rebuilt and compared again:

- `structured_winner`
- `ridge_direct_full`
- `histgb_pruned_v1_no_anchor_no_regime`
- `hybrid_slice_no_hard_case_override_pruned`

The hybrid routing logic remained fixed:

- default to `histgb_pruned_v1_no_anchor_no_regime`
- use `structured_winner` on:
  - center
  - `quadratic_smile_only`
- use `ridge_direct_full` on:
  - `tv_maturity_only`

No routing redesign was performed in this notebook.

---

## 4. Final tuned ranking

The final tuned leader from this notebook is:

- `hybrid_slice_no_hard_case_override_pruned`

Its tuned protocol-level performance is:

- `primary_realistic = 0.8125`
- `stress_test = 0.8154`
- `avg_two_protocols = 0.8139`
- `max_two_protocols = 0.8154`

This remains clearly ahead of the standalone challengers:

- `histgb_pruned_v1_no_anchor_no_regime`
  - `0.8725 / 0.9922`

- `ridge_direct_full`
  - `0.8747 / 0.8898`

- `structured_winner`
  - `1.0638 / 1.1847`

So the hybrid still wins after:

- replacing the nonlinear branch with the pruned HistGB model
- choosing a shared training policy
- and doing narrow finalist-only tuning

---

## 5. Effect of tuning

### Hybrid
Tuning produced:
- a small improvement on `primary_realistic`
- essentially flat performance on `stress_test`

So tuning helped slightly, but did not change the identity of the leader.

### Ridge
Ridge benefited meaningfully from stronger regularization.
It remains the strongest simple linear challenger, but still trails the hybrid.

### Pruned HistGB
The pruned HistGB baseline configuration remained the best nonlinear standalone setup.
No tuned variant improved it enough to change the ranking.

---

## 6. Final interpretation from 06.1

This notebook confirms three things:

1. the best shared ML training policy is:
   - `primary_only`
2. the leading hybrid remains the best overall candidate after narrow tuning
3. the final model landscape is now stable enough to move to final inference packaging

The simplicity-bias rule does **not** activate here because the leading hybrid is not merely marginally better than the standalone challengers; it remains clearly ahead.

---

## 7. Model advancing to 06.2

The single model advancing to final inference and submission checks is:

- `hybrid_slice_no_hard_case_override_pruned`

with:

### Shared training policy
- `primary_only`

### Ridge branch
- `alpha = 10.0`

### Pruned HistGB branch
- `learning_rate = 0.05`
- `max_depth = 3`
- `max_iter = 300`
- `min_samples_leaf = 10`

This is the final locked model entering `06_2_final_inference_and_submission_checks.ipynb`.

---

## 8. Main conclusion

The Phase 6 workflow now has a fully selected and tuned finalist.

The project can move on from model selection to final operational work:

- fit the locked model on the full training data
- build test-time features for the true missing rows
- run final inference
- inspect route mix, maturity mix, wing/center mix, and prediction sanity
- and prepare final submission artifacts


## 6.1 - Updates

In [50]:
PHASE61_TOP3_MODELS = [
    "hybrid_slice_no_hard_case_override_pruned",
    "histgb_pruned_v1_no_anchor_no_regime",
    "ridge_direct_full",
]

phase61_diag_rows = []

for eval_protocol in PROTOCOLS:
    for fold_id in fold_plan["fold"].astype(int).tolist():
        val_table = validation_table_cache[(eval_protocol, fold_id)].copy()

        base = val_table[
            [
                "row_id",
                "target_date",
                "target_iv",
                "maturity_label",
                "maturity_days",
                "option_type",
                "moneyness",
                "wing_center_bucket",
                "hard_case",
                "true_local_support_score",
                "structured_winner_source",
            ]
        ].copy()

        base["hard_case_bucket"] = np.where(base["hard_case"], "hard_case", "non_hard_case")

        for model_name in PHASE61_TOP3_MODELS:
            pred_df = phase61_tuned_prediction_cache[(eval_protocol, fold_id, model_name)].copy()

            pred_cols = ["row_id", "iv_pred"]
            if model_name == "hybrid_slice_no_hard_case_override_pruned":
                pred_cols.append("hybrid_route")

            merged = base.merge(pred_df[pred_cols], on="row_id", how="left")
            merged["eval_protocol"] = eval_protocol
            merged["fold"] = fold_id
            merged["model"] = model_name
            merged["abs_error"] = (merged["target_iv"] - merged["iv_pred"]).abs()
            merged["sq_error"] = (merged["target_iv"] - merged["iv_pred"]) ** 2

            if "hybrid_route" not in merged.columns:
                merged["hybrid_route"] = pd.NA

            phase61_diag_rows.append(merged)

phase61_diag_df = pd.concat(phase61_diag_rows, ignore_index=True)

print("06.1 tuned diagnostics dataframe")
display(phase61_diag_df.head())


06.1 tuned diagnostics dataframe


,row_id,target_date,target_iv,maturity_label,maturity_days,option_type,moneyness,wing_center_bucket,hard_case,true_local_support_score,structured_winner_source,hard_case_bucket,iv_pred,hybrid_route,eval_protocol,fold,model,abs_error,sq_error
0,9262,2025-04-21,19.2304,1M,30,call,1.1000,center,False,4,structured_region_blend_callput_shrink,non_hard_case,19.1951,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.0353,0.0012
1,9245,2025-04-21,24.2840,1M,30,put,0.8750,wing,False,3,quadratic_smile_only,non_hard_case,25.1451,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.8611,0.7415
2,9294,2025-04-21,19.0918,2M,60,call,1.1250,wing,False,5,structured_region_blend_callput_shrink,non_hard_case,18.2708,histgb_pruned_v1_no_anchor_no_regime,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.8210,0.6740
3,9281,2025-04-21,20.2648,2M,60,put,0.9500,center,False,4,structured_region_blend_callput_shrink,non_hard_case,20.5633,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.2985,0.0891
4,9312,2025-04-21,18.7062,3M,91,call,0.9750,center,False,4,structured_region_blend_callput_shrink,non_hard_case,19.3098,structured_winner,primary_realistic,1,hybrid_slice_no_hard_case_override_pruned,0.6036,0.3643


In [51]:
def slice_error_table_phase61(df: pd.DataFrame, group_cols) -> pd.DataFrame:
    group_cols = list(group_cols)

    out = (
        df.groupby(["eval_protocol", "model", *group_cols], dropna=False)
        .agg(
            n=("row_id", "size"),
            mae=("abs_error", "mean"),
            mean_sq_error=("sq_error", "mean"),
        )
        .reset_index()
    )

    out["rmse"] = np.sqrt(out["mean_sq_error"])
    return out.drop(columns=["mean_sq_error"]).sort_values(
        ["eval_protocol", "model", *group_cols]
    ).reset_index(drop=True)


def slice_error_table_phase61_with_fold(df: pd.DataFrame, group_cols) -> pd.DataFrame:
    group_cols = list(group_cols)

    out = (
        df.groupby(["eval_protocol", "fold", "model", *group_cols], dropna=False)
        .agg(
            n=("row_id", "size"),
            mae=("abs_error", "mean"),
            mean_sq_error=("sq_error", "mean"),
        )
        .reset_index()
    )

    out["rmse"] = np.sqrt(out["mean_sq_error"])
    return out.drop(columns=["mean_sq_error"]).sort_values(
        ["eval_protocol", "fold", "model", *group_cols]
    ).reset_index(drop=True)


In [52]:
phase61_top3_wing_center = slice_error_table_phase61(
    phase61_diag_df,
    group_cols=["wing_center_bucket"],
)

phase61_top3_maturity = slice_error_table_phase61(
    phase61_diag_df,
    group_cols=["maturity_label"],
)

phase61_top3_hard_case = slice_error_table_phase61(
    phase61_diag_df,
    group_cols=["hard_case_bucket"],
)

print("06.1 top-3 finalists | wing vs center")
display(phase61_top3_wing_center)

print("06.1 top-3 finalists | maturity")
display(phase61_top3_maturity)

print("06.1 top-3 finalists | hard-case bucket")
display(phase61_top3_hard_case)


06.1 top-3 finalists | wing vs center


,eval_protocol,model,wing_center_bucket,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,center,65,0.6043,0.7645
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,wing,91,0.7871,0.9439
2,primary_realistic,hybrid_slice_no_hard_case_override_pruned,center,65,0.5069,0.6537
3,primary_realistic,hybrid_slice_no_hard_case_override_pruned,wing,91,0.7573,0.9109
4,primary_realistic,ridge_direct_full,center,65,0.5653,0.7270
5,primary_realistic,ridge_direct_full,wing,91,0.7651,0.9715
6,stress_test,histgb_pruned_v1_no_anchor_no_regime,center,49,0.6186,0.7681
7,stress_test,histgb_pruned_v1_no_anchor_no_regime,wing,107,0.8851,1.0850
8,stress_test,hybrid_slice_no_hard_case_override_pruned,center,49,0.5279,0.6479
9,stress_test,hybrid_slice_no_hard_case_override_pruned,wing,107,0.7170,0.8916


06.1 top-3 finalists | maturity


,eval_protocol,model,maturity_label,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,1M,40,0.7523,0.9516
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,2M,39,0.7090,0.8688
2,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,3M,39,0.7484,0.8758
3,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,6M,38,0.6312,0.7862
4,primary_realistic,hybrid_slice_no_hard_case_override_pruned,1M,40,0.6375,0.8477
5,primary_realistic,hybrid_slice_no_hard_case_override_pruned,2M,39,0.7087,0.8698
6,primary_realistic,hybrid_slice_no_hard_case_override_pruned,3M,39,0.6735,0.7805
7,primary_realistic,hybrid_slice_no_hard_case_override_pruned,6M,38,0.5911,0.7483
8,primary_realistic,ridge_direct_full,1M,40,0.7490,1.0221
9,primary_realistic,ridge_direct_full,2M,39,0.6308,0.8867


06.1 top-3 finalists | hard-case bucket


,eval_protocol,model,hard_case_bucket,n,mae,rmse
0,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,hard_case,6,1.1577,1.4627
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,non_hard_case,150,0.6931,0.8415
2,primary_realistic,hybrid_slice_no_hard_case_override_pruned,hard_case,6,1.1577,1.4627
3,primary_realistic,hybrid_slice_no_hard_case_override_pruned,non_hard_case,150,0.6328,0.7765
4,primary_realistic,ridge_direct_full,hard_case,6,1.4856,1.8912
5,primary_realistic,ridge_direct_full,non_hard_case,150,0.6497,0.8115
6,stress_test,histgb_pruned_v1_no_anchor_no_regime,hard_case,12,0.9360,1.1035
7,stress_test,histgb_pruned_v1_no_anchor_no_regime,non_hard_case,144,0.7901,0.9869
8,stress_test,hybrid_slice_no_hard_case_override_pruned,hard_case,12,0.7917,0.9196
9,stress_test,hybrid_slice_no_hard_case_override_pruned,non_hard_case,144,0.6465,0.8143


In [53]:
phase61_tuned_hybrid_route_mix = (
    phase61_diag_df.loc[
        phase61_diag_df["model"] == "hybrid_slice_no_hard_case_override_pruned"
    ]
    .groupby(["eval_protocol", "hybrid_route"], as_index=False)["row_id"]
    .count()
    .rename(columns={"row_id": "count"})
    .sort_values(["eval_protocol", "count"], ascending=[True, False])
    .reset_index(drop=True)
)

print("06.1 tuned hybrid route mix")
display(phase61_tuned_hybrid_route_mix)


06.1 tuned hybrid route mix


,eval_protocol,hybrid_route,count
0,primary_realistic,structured_winner,88
1,primary_realistic,histgb_pruned_v1_no_anchor_no_regime,58
2,primary_realistic,ridge_direct_full,10
3,stress_test,structured_winner,74
4,stress_test,histgb_pruned_v1_no_anchor_no_regime,69
5,stress_test,ridge_direct_full,13


In [54]:
WEAK_FOLD_KEYS = pd.DataFrame(
    [
        {"eval_protocol": "primary_realistic", "fold": 2},
        {"eval_protocol": "primary_realistic", "fold": 4},
        {"eval_protocol": "stress_test", "fold": 4},
    ]
)

phase61_weak_fold_df = (
    phase61_diag_df
    .merge(WEAK_FOLD_KEYS, on=["eval_protocol", "fold"], how="inner")
    .copy()
)

print("06.1 weak-fold audit dataframe")
display(
    phase61_weak_fold_df[
        ["eval_protocol", "fold", "model", "row_id", "maturity_label", "wing_center_bucket", "hard_case_bucket", "structured_winner_source", "hybrid_route"]
    ].head()
)


06.1 weak-fold audit dataframe


,eval_protocol,fold,model,row_id,maturity_label,wing_center_bucket,hard_case_bucket,structured_winner_source,hybrid_route
0,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,9866,1M,wing,hard_case,same_date_linear_interp,histgb_pruned_v1_no_anchor_no_regime
1,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,9867,1M,wing,non_hard_case,quadratic_smile_only,structured_winner
2,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,9896,2M,wing,hard_case,same_date_linear_interp,histgb_pruned_v1_no_anchor_no_regime
3,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,9871,2M,wing,non_hard_case,structured_region_blend_callput_shrink,histgb_pruned_v1_no_anchor_no_regime
4,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,9926,3M,wing,non_hard_case,structured_region_blend_callput_shrink,histgb_pruned_v1_no_anchor_no_regime


In [55]:
phase61_weak_fold_wing_center = slice_error_table_phase61_with_fold(
    phase61_weak_fold_df,
    group_cols=["wing_center_bucket"],
)

phase61_weak_fold_maturity = slice_error_table_phase61_with_fold(
    phase61_weak_fold_df,
    group_cols=["maturity_label"],
)

phase61_weak_fold_hard_case = slice_error_table_phase61_with_fold(
    phase61_weak_fold_df,
    group_cols=["hard_case_bucket"],
)

print("06.1 weak-fold audit | wing vs center")
display(phase61_weak_fold_wing_center)

print("06.1 weak-fold audit | maturity")
display(phase61_weak_fold_maturity)

print("06.1 weak-fold audit | hard-case bucket")
display(phase61_weak_fold_hard_case)


06.1 weak-fold audit | wing vs center


,eval_protocol,fold,model,wing_center_bucket,n,mae,rmse
0,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,center,10,0.5173,0.6643
1,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,wing,28,0.8335,1.0346
2,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,center,10,0.4271,0.4667
3,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,wing,28,0.8130,1.0116
4,primary_realistic,2,ridge_direct_full,center,10,0.4666,0.5516
5,primary_realistic,2,ridge_direct_full,wing,28,0.7768,0.9605
6,primary_realistic,4,histgb_pruned_v1_no_anchor_no_regime,center,21,0.6508,0.8457
7,primary_realistic,4,histgb_pruned_v1_no_anchor_no_regime,wing,19,0.7921,0.9230
8,primary_realistic,4,hybrid_slice_no_hard_case_override_pruned,center,21,0.4739,0.7162
9,primary_realistic,4,hybrid_slice_no_hard_case_override_pruned,wing,19,0.7682,0.9014


06.1 weak-fold audit | maturity


,eval_protocol,fold,model,maturity_label,n,mae,rmse
0,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,1M,10,0.8149,1.0752
1,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,2M,10,0.4584,0.6848
2,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,3M,9,0.9071,1.0469
3,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,6M,9,0.8461,0.9585
4,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,1M,10,0.7801,1.0788
5,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,2M,10,0.6287,0.8219
6,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,3M,9,0.6756,0.8173
7,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,6M,9,0.7632,0.8450
8,primary_realistic,2,ridge_direct_full,1M,10,0.8520,1.1728
9,primary_realistic,2,ridge_direct_full,2M,10,0.5700,0.7104


06.1 weak-fold audit | hard-case bucket


,eval_protocol,fold,model,hard_case_bucket,n,mae,rmse
0,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,hard_case,2,1.3557,1.8476
1,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,non_hard_case,36,0.7167,0.8749
2,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,hard_case,2,1.3557,1.8476
3,primary_realistic,2,hybrid_slice_no_hard_case_override_pruned,non_hard_case,36,0.6757,0.8165
4,primary_realistic,2,ridge_direct_full,hard_case,2,2.0964,2.2670
5,primary_realistic,2,ridge_direct_full,non_hard_case,36,0.6173,0.7187
6,primary_realistic,4,histgb_pruned_v1_no_anchor_no_regime,hard_case,1,1.0002,1.0002
7,primary_realistic,4,histgb_pruned_v1_no_anchor_no_regime,non_hard_case,39,0.7107,0.8800
8,primary_realistic,4,hybrid_slice_no_hard_case_override_pruned,hard_case,1,1.0002,1.0002
9,primary_realistic,4,hybrid_slice_no_hard_case_override_pruned,non_hard_case,39,0.6037,0.8040


In [56]:
phase61_weak_fold_hybrid_route_mix = (
    phase61_weak_fold_df.loc[
        phase61_weak_fold_df["model"] == "hybrid_slice_no_hard_case_override_pruned"
    ]
    .groupby(["eval_protocol", "fold", "hybrid_route"], as_index=False)["row_id"]
    .count()
    .rename(columns={"row_id": "count"})
    .sort_values(["eval_protocol", "fold", "count"], ascending=[True, True, False])
    .reset_index(drop=True)
)

phase61_weak_fold_hybrid_source_mix = (
    phase61_weak_fold_df.loc[
        phase61_weak_fold_df["model"] == "hybrid_slice_no_hard_case_override_pruned"
    ]
    .groupby(["eval_protocol", "fold", "structured_winner_source"], as_index=False)["row_id"]
    .count()
    .rename(columns={"row_id": "count"})
    .sort_values(["eval_protocol", "fold", "count"], ascending=[True, True, False])
    .reset_index(drop=True)
)

print("06.1 weak-fold audit | hybrid route mix")
display(phase61_weak_fold_hybrid_route_mix)

print("06.1 weak-fold audit | hybrid source mix")
display(phase61_weak_fold_hybrid_source_mix)


06.1 weak-fold audit | hybrid route mix


,eval_protocol,fold,hybrid_route,count
0,primary_realistic,2,structured_winner,21
1,primary_realistic,2,histgb_pruned_v1_no_anchor_no_regime,14
2,primary_realistic,2,ridge_direct_full,3
3,primary_realistic,4,structured_winner,23
4,primary_realistic,4,histgb_pruned_v1_no_anchor_no_regime,15
5,primary_realistic,4,ridge_direct_full,2
6,stress_test,4,histgb_pruned_v1_no_anchor_no_regime,20
7,stress_test,4,structured_winner,17
8,stress_test,4,ridge_direct_full,3


06.1 weak-fold audit | hybrid source mix


,eval_protocol,fold,structured_winner_source,count
0,primary_realistic,2,structured_region_blend_callput_shrink,17
1,primary_realistic,2,quadratic_smile_only,13
2,primary_realistic,2,same_date_linear_interp,3
3,primary_realistic,2,tv_maturity_only,3
4,primary_realistic,2,structured_region_blend_center,1
5,primary_realistic,2,structured_region_blend_wing,1
6,primary_realistic,4,structured_region_blend_callput_shrink,30
7,primary_realistic,4,same_date_linear_interp,4
8,primary_realistic,4,quadratic_smile_only,3
9,primary_realistic,4,tv_maturity_only,2


In [57]:
HYBRID_SENSITIVITY_CONFIGS = pd.DataFrame(
    [
        {
            "label": "locked_r10_hgb_base",
            "ridge_config_id": "ridge_a_10p0",
            "histgb_config_id": "hgb_base",
        },
        {
            "label": "alt_r3_hgb_base",
            "ridge_config_id": "ridge_a_3p0",
            "histgb_config_id": "hgb_base",
        },
        {
            "label": "alt_r10_hgb_depth2",
            "ridge_config_id": "ridge_a_10p0",
            "histgb_config_id": "hgb_depth_2",
        },
        {
            "label": "alt_r3_hgb_depth2",
            "ridge_config_id": "ridge_a_3p0",
            "histgb_config_id": "hgb_depth_2",
        },
    ]
)

print("06.1 hybrid branch-sensitivity configs")
display(HYBRID_SENSITIVITY_CONFIGS)


06.1 hybrid branch-sensitivity configs


,label,ridge_config_id,histgb_config_id
0,locked_r10_hgb_base,ridge_a_10p0,hgb_base
1,alt_r3_hgb_base,ridge_a_3p0,hgb_base
2,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2
3,alt_r3_hgb_depth2,ridge_a_3p0,hgb_depth_2


In [58]:
phase61_hybrid_sensitivity_rows = []

for _, cfg in HYBRID_SENSITIVITY_CONFIGS.iterrows():
    label = cfg["label"]
    ridge_config_id = cfg["ridge_config_id"]
    histgb_config_id = cfg["histgb_config_id"]

    for eval_protocol in PROTOCOLS:
        for fold_id in fold_plan["fold"].astype(int).tolist():
            val_table = validation_table_cache[(eval_protocol, fold_id)]

            structured_pred_df = val_table[["row_id", "structured_winner_pred"]].rename(
                columns={"structured_winner_pred": "iv_pred"}
            )
            ridge_pred_df = ridge_tuning_prediction_cache[(ridge_config_id, eval_protocol, fold_id)]
            histgb_pred_df = histgb_tuning_prediction_cache[(histgb_config_id, eval_protocol, fold_id)]

            panel_df = build_prediction_panel_for_policy_eval(
                val_table,
                structured_pred_df=structured_pred_df,
                ridge_pred_df=ridge_pred_df,
                pruned_histgb_pred_df=histgb_pred_df,
            )

            hybrid_pred_df = apply_hybrid_slice_no_hard_case_override_pruned(panel_df).copy()
            hybrid_rmse = rmse_from_series(hybrid_pred_df["target_iv"], hybrid_pred_df["iv_pred"])

            phase61_hybrid_sensitivity_rows.append(
                {
                    "label": label,
                    "ridge_config_id": ridge_config_id,
                    "histgb_config_id": histgb_config_id,
                    "eval_protocol": eval_protocol,
                    "fold": fold_id,
                    "val_rmse": hybrid_rmse,
                }
            )

phase61_hybrid_sensitivity_results = (
    pd.DataFrame(phase61_hybrid_sensitivity_rows)
    .sort_values(["label", "eval_protocol", "fold"])
    .reset_index(drop=True)
)

print("06.1 hybrid branch-sensitivity results")
display(phase61_hybrid_sensitivity_results)


06.1 hybrid branch-sensitivity results


,label,ridge_config_id,histgb_config_id,eval_protocol,fold,val_rmse
0,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,primary_realistic,1,0.8313
1,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,primary_realistic,2,0.9097
2,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,primary_realistic,3,0.7856
3,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,primary_realistic,4,0.8040
4,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,stress_test,1,0.8544
5,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,stress_test,2,0.7444
6,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,stress_test,3,0.7185
7,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,stress_test,4,0.9385
8,alt_r3_hgb_base,ridge_a_3p0,hgb_base,primary_realistic,1,0.7569
9,alt_r3_hgb_base,ridge_a_3p0,hgb_base,primary_realistic,2,0.9026


In [59]:
phase61_hybrid_sensitivity_summary = (
    phase61_hybrid_sensitivity_results
    .groupby(["label", "ridge_config_id", "histgb_config_id", "eval_protocol"])
    .agg(
        mean_rmse=("val_rmse", "mean"),
        min_rmse=("val_rmse", "min"),
        max_rmse=("val_rmse", "max"),
    )
    .reset_index()
)

phase61_hybrid_sensitivity_pivot = (
    phase61_hybrid_sensitivity_summary
    .pivot(
        index=["label", "ridge_config_id", "histgb_config_id"],
        columns="eval_protocol",
        values="mean_rmse",
    )
    .reset_index()
)

phase61_hybrid_sensitivity_pivot["avg_two_protocols"] = (
    phase61_hybrid_sensitivity_pivot["primary_realistic"] + phase61_hybrid_sensitivity_pivot["stress_test"]
) / 2.0

phase61_hybrid_sensitivity_pivot["max_two_protocols"] = phase61_hybrid_sensitivity_pivot[
    ["primary_realistic", "stress_test"]
].max(axis=1)

print("06.1 hybrid branch-sensitivity pivot")
display(
    phase61_hybrid_sensitivity_pivot.sort_values(
        ["primary_realistic", "max_two_protocols", "avg_two_protocols"]
    )
)


06.1 hybrid branch-sensitivity pivot


eval_protocol,label,ridge_config_id,histgb_config_id,primary_realistic,stress_test,avg_two_protocols,max_two_protocols
3,locked_r10_hgb_base,ridge_a_10p0,hgb_base,0.8125,0.8154,0.8139,0.8154
1,alt_r3_hgb_base,ridge_a_3p0,hgb_base,0.8137,0.8148,0.8143,0.8148
0,alt_r10_hgb_depth2,ridge_a_10p0,hgb_depth_2,0.8326,0.8140,0.8233,0.8326
2,alt_r3_hgb_depth2,ridge_a_3p0,hgb_depth_2,0.8339,0.8134,0.8236,0.8339


## Phase 6.2 Final Inference and Submission Checks

This section takes the final locked model from Phase 6.1 and applies it to the true missing rows in the test set.

The locked final model is:

- `hybrid_slice_no_hard_case_override_pruned`

with:

- shared training policy: `primary_only`
- Ridge branch:
  - `alpha = 10.0`
- pruned HistGB branch:
  - `learning_rate = 0.05`
  - `max_depth = 3`
  - `max_iter = 300`
  - `min_samples_leaf = 10`

This section will:

1. build the full final pseudo-training table under the locked policy
2. fit the final Ridge and pruned HistGB branches
3. build the feature table for the true missing test rows
4. generate final hybrid predictions
5. run final sanity checks:
   - route mix
   - source mix
   - maturity mix
   - wing/center mix
   - final prediction vs structured baseline by slice
   - total variance sanity


In [60]:
FINAL_MODEL_NAME = "hybrid_slice_no_hard_case_override_pruned"
FINAL_SHARED_POLICY = "primary_only"

FINAL_RIDGE_ALPHA = 10.0

FINAL_HISTGB_CONFIG = {
    "learning_rate": 0.05,
    "max_depth": 3,
    "max_iter": 300,
    "min_samples_leaf": 10,
}

print("Final model:", FINAL_MODEL_NAME)
print("Final shared policy:", FINAL_SHARED_POLICY)
print("Final Ridge alpha:", FINAL_RIDGE_ALPHA)
print("Final HistGB config:", FINAL_HISTGB_CONFIG)


Final model: hybrid_slice_no_hard_case_override_pruned
Final shared policy: primary_only
Final Ridge alpha: 10.0
Final HistGB config: {'learning_rate': 0.05, 'max_depth': 3, 'max_iter': 300, 'min_samples_leaf': 10}


In [66]:
def build_full_training_table_for_protocol(
    train_df: pd.DataFrame,
    protocol_name: str,
    lookback_dates: int = 20,
    min_history_dates: int = ML_MIN_HISTORY_DATES,
) -> pd.DataFrame:
    target_dates = train_date_summary.index.to_list()[min_history_dates:]
    tables = []

    for target_date in target_dates:
        bundle = make_single_date_training_bundle(
            train_df,
            target_date=target_date,
            protocol_name=protocol_name,
        )

        feat = build_feature_table_for_masked_bundle(
            bundle["train_history"],
            bundle["masked_target_date"],
            lookback_dates=lookback_dates,
        )

        scored = feat.loc[feat["is_scored_target"]].copy()
        if not scored["is_orig_observed"].all():
            raise ValueError("Final pseudo-training rows must come from originally observed train rows only.")

        scored["target_date"] = pd.Timestamp(target_date)
        scored["protocol"] = protocol_name
        scored["n_history_dates"] = bundle["train_history"]["date"].nunique()
        tables.append(scored)

    return pd.concat(tables, ignore_index=True)


def build_full_training_table_for_policy(
    train_df: pd.DataFrame,
    training_policy: str,
    lookback_dates: int = 20,
) -> pd.DataFrame:
    if training_policy == "primary_only":
        policy_protocols = ["primary_realistic"]
    elif training_policy == "stress_only":
        policy_protocols = ["stress_test"]
    elif training_policy == "union_shared":
        policy_protocols = ["primary_realistic", "stress_test"]
    else:
        raise ValueError(f"Unknown training_policy: {training_policy}")

    tables = []
    for protocol_name in policy_protocols:
        table = build_full_training_table_for_protocol(
            train_df,
            protocol_name=protocol_name,
            lookback_dates=lookback_dates,
        ).copy()
        table["training_policy"] = training_policy
        table["source_protocol_for_training_rows"] = protocol_name
        tables.append(table)

    return pd.concat(tables, ignore_index=True)


def build_actual_test_masked_rows(test_df: pd.DataFrame) -> pd.DataFrame:
    out = test_df.copy()

    out["is_orig_observed"] = out["iv_observed"].notna()
    out["is_orig_missing"] = ~out["is_orig_observed"]

    out["is_pseudo_hidden"] = False
    out["is_scored_target"] = out["is_orig_missing"]
    out["is_visible_anchor"] = out["is_orig_observed"]
    out["is_effectively_missing"] = out["is_orig_missing"]
    out["mask_protocol"] = "actual_test_missing"

    # For true test inference there is no pseudo-mask selection stage, but the ML
    # feature schema still expects mask_local_support_score downstream.
    # Use the actual same-date visible support as the mask-side proxy so:
    # - mask_local_support_score exists
    # - support_score_gap stays well-defined
    # - train/test feature schemas remain aligned
    scored_targets = out.loc[out["is_scored_target"]].copy()
    visible_anchors = out.loc[out["is_visible_anchor"]].copy()

    if len(scored_targets) > 0:
        support = local_support_profile(
            scored_targets,
            visible_anchors,
        )[["row_id", "local_support_score"]].copy()

        out = out.merge(support, on="row_id", how="left")
    else:
        out["local_support_score"] = np.nan

    out["local_support_score"] = out["local_support_score"].fillna(0).astype(int)

    return out.sort_values(["date", "maturity_days", "option_type", "moneyness"]).reset_index(drop=True)



In [67]:
final_full_train_table = build_full_training_table_for_policy(
    train,
    training_policy=FINAL_SHARED_POLICY,
    lookback_dates=20,
)

print("06.2 final full training table summary")
display(summarize_feature_table(final_full_train_table).to_frame("value"))

print("06.2 final full training table protocol mix")
display(
    final_full_train_table["source_protocol_for_training_rows"]
    .value_counts()
    .rename_axis("source_protocol_for_training_rows")
    .to_frame("count")
)


06.2 final full training table summary


,value
n_rows,592
n_dates,77
date_min,2025-01-30
date_max,2025-05-16
target_iv_missing,0
target_residual_missing,0
winner_pred_missing,0
mean_target_iv,19.9346
mean_target_residual,0.0702
rmse_structured_winner,0.9538


06.2 final full training table protocol mix


,count
source_protocol_for_training_rows,
primary_realistic,592


In [68]:
actual_test_masked_rows = build_actual_test_masked_rows(test)

full_test_feature_table = build_feature_table_for_masked_bundle(
    train_history=train.copy(),
    masked_target_rows=actual_test_masked_rows,
    lookback_dates=20,
)

test_missing_feature_table = full_test_feature_table.loc[
    full_test_feature_table["is_scored_target"]
].copy().reset_index(drop=True)

print("06.2 test missing-row feature table summary")
display(
    pd.Series(
        {
            "n_test_rows_total": len(test),
            "n_test_missing_rows": len(test_missing_feature_table),
            "n_test_dates": test_missing_feature_table["date"].nunique(),
            "winner_pred_missing": int(test_missing_feature_table["structured_winner_pred"].isna().sum()),
            "recent_node_pred_missing": int(test_missing_feature_table["recent_node_pred"].isna().sum()),
            "full_node_pred_missing": int(test_missing_feature_table["full_node_pred"].isna().sum()),
        }
    ).to_frame("value")
)


06.2 test missing-row feature table summary


,value
n_test_rows_total,3960
n_test_missing_rows,1699
n_test_dates,33
winner_pred_missing,0
recent_node_pred_missing,0
full_node_pred_missing,0


In [69]:
missing_full_cols = sorted(set(FULL_HISTGB_ABLATION_FEATURES) - set(test_missing_feature_table.columns))
missing_pruned_cols = sorted(set(PHASE6_PRUNED_HISTGB_FEATURES) - set(test_missing_feature_table.columns))

print("Missing FULL_HISTGB_ABLATION_FEATURES columns on test:", missing_full_cols)
print("Missing PHASE6_PRUNED_HISTGB_FEATURES columns on test:", missing_pruned_cols)


Missing FULL_HISTGB_ABLATION_FEATURES columns on test: []
Missing PHASE6_PRUNED_HISTGB_FEATURES columns on test: []


In [72]:
def build_prediction_panel_for_final_inference(
    test_feature_table: pd.DataFrame,
    structured_pred_df: pd.DataFrame,
    ridge_pred_df: pd.DataFrame,
    pruned_histgb_pred_df: pd.DataFrame,
) -> pd.DataFrame:
    base = test_feature_table[
        [
            "row_id",
            "date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "moneyness",
            "wing_center_bucket",
            "hard_case",
            "structured_winner_source",
        ]
    ].copy()

    structured_pred = structured_pred_df[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "structured_winner__iv_pred"}
    )
    ridge_pred = ridge_pred_df[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "ridge_direct_full__iv_pred"}
    )
    pruned_histgb_pred = pruned_histgb_pred_df[["row_id", "iv_pred"]].rename(
        columns={"iv_pred": "histgb_pruned_v1_no_anchor_no_regime__iv_pred"}
    )

    panel = base.merge(structured_pred, on="row_id", how="left")
    panel = panel.merge(ridge_pred, on="row_id", how="left")
    panel = panel.merge(pruned_histgb_pred, on="row_id", how="left")

    return panel


In [70]:
# Ridge branch
X_train_ridge, X_test_ridge = prepare_design_matrices(
    final_full_train_table,
    test_missing_feature_table,
    FULL_HISTGB_ABLATION_FEATURES,
)

final_ridge_model = Ridge(alpha=FINAL_RIDGE_ALPHA)
final_ridge_model.fit(X_train_ridge, final_full_train_table["target_iv"])

final_ridge_pred_df = test_missing_feature_table[["row_id"]].copy()
final_ridge_pred_df["iv_pred"] = final_ridge_model.predict(X_test_ridge)


# Pruned HistGB residual branch
X_train_hgb, X_test_hgb = prepare_design_matrices(
    final_full_train_table,
    test_missing_feature_table,
    PHASE6_PRUNED_HISTGB_FEATURES,
)

final_histgb_model = HistGradientBoostingRegressor(
    learning_rate=FINAL_HISTGB_CONFIG["learning_rate"],
    max_depth=FINAL_HISTGB_CONFIG["max_depth"],
    max_iter=FINAL_HISTGB_CONFIG["max_iter"],
    min_samples_leaf=FINAL_HISTGB_CONFIG["min_samples_leaf"],
    random_state=42,
)
final_histgb_model.fit(X_train_hgb, final_full_train_table["target_residual"])

final_histgb_pred_df = test_missing_feature_table[
    ["row_id", "structured_winner_pred"]
].copy()
final_histgb_pred_df["ml_raw_pred"] = final_histgb_model.predict(X_test_hgb)
final_histgb_pred_df["iv_pred"] = (
    final_histgb_pred_df["structured_winner_pred"] + final_histgb_pred_df["ml_raw_pred"]
)


# Structured branch
final_structured_pred_df = test_missing_feature_table[["row_id", "structured_winner_pred"]].rename(
    columns={"structured_winner_pred": "iv_pred"}
)


In [73]:
final_test_panel = build_prediction_panel_for_final_inference(
    test_missing_feature_table,
    structured_pred_df=final_structured_pred_df,
    ridge_pred_df=final_ridge_pred_df,
    pruned_histgb_pred_df=final_histgb_pred_df[["row_id", "iv_pred"]],
)

final_test_hybrid_preds = apply_hybrid_slice_no_hard_case_override_pruned(
    final_test_panel
).copy()

final_test_predictions = (
    test_missing_feature_table[
        [
            "row_id",
            "date",
            "option_type",
            "maturity_label",
            "maturity_days",
            "tau",
            "moneyness",
            "wing_center_bucket",
            "hard_case",
            "structured_winner_source",
            "structured_winner_pred",
        ]
    ]
    .merge(
        final_ridge_pred_df.rename(columns={"iv_pred": "ridge_direct_full_pred"}),
        on="row_id",
        how="left",
    )
    .merge(
        final_histgb_pred_df[["row_id", "iv_pred"]].rename(
            columns={"iv_pred": "histgb_pruned_v1_no_anchor_no_regime_pred"}
        ),
        on="row_id",
        how="left",
    )
    .merge(
        final_test_hybrid_preds[["row_id", "hybrid_route", "iv_pred"]].rename(
            columns={"iv_pred": "final_iv_pred"}
        ),
        on="row_id",
        how="left",
    )
)

final_test_predictions["delta_vs_structured"] = (
    final_test_predictions["final_iv_pred"] - final_test_predictions["structured_winner_pred"]
)
final_test_predictions["abs_delta_vs_structured"] = final_test_predictions["delta_vs_structured"].abs()
final_test_predictions["pred_total_variance"] = (
    (final_test_predictions["final_iv_pred"] / 100.0) ** 2 * final_test_predictions["tau"]
)

print("06.2 final test prediction table preview")
display(final_test_predictions.head())


06.2 final test prediction table preview


,row_id,date,option_type,maturity_label,maturity_days,tau,moneyness,wing_center_bucket,hard_case,structured_winner_source,structured_winner_pred,ridge_direct_full_pred,histgb_pruned_v1_no_anchor_no_regime_pred,hybrid_route,final_iv_pred,delta_vs_structured,abs_delta_vs_structured,pred_total_variance
0,11648,2025-05-19,call,1M,30,0.1191,0.9250,center,False,structured_region_blend_callput_shrink,17.5375,18.0100,17.4191,structured_winner,17.5375,0.0000,0.0000,0.0037
1,11650,2025-05-19,call,1M,30,0.1191,0.9500,center,False,quadratic_smile_only,16.7037,17.4770,16.9173,structured_winner,16.7037,0.0000,0.0000,0.0033
2,11656,2025-05-19,call,1M,30,0.1191,1.0250,center,False,structured_region_blend_callput_shrink,15.6926,16.1604,15.5789,structured_winner,15.6926,0.0000,0.0000,0.0029
3,11658,2025-05-19,call,1M,30,0.1191,1.0500,center,False,structured_region_blend_callput_shrink,15.2995,15.7369,15.1689,structured_winner,15.2995,0.0000,0.0000,0.0028
4,11662,2025-05-19,call,1M,30,0.1191,1.1000,center,False,quadratic_smile_only,14.9134,15.4786,15.2225,structured_winner,14.9134,0.0000,0.0000,0.0026


In [74]:
print("06.2 final prediction range sanity")
display(
    pd.Series(
        {
            "n_predictions": len(final_test_predictions),
            "n_nonpositive_predictions": int((final_test_predictions["final_iv_pred"] <= 0).sum()),
            "min_final_iv_pred": float(final_test_predictions["final_iv_pred"].min()),
            "p01_final_iv_pred": float(final_test_predictions["final_iv_pred"].quantile(0.01)),
            "median_final_iv_pred": float(final_test_predictions["final_iv_pred"].median()),
            "p99_final_iv_pred": float(final_test_predictions["final_iv_pred"].quantile(0.99)),
            "max_final_iv_pred": float(final_test_predictions["final_iv_pred"].max()),
        }
    ).to_frame("value")
)


06.2 final prediction range sanity


,value
n_predictions,"1,699.0000"
n_nonpositive_predictions,0.0000
min_final_iv_pred,12.7884
p01_final_iv_pred,13.6089
median_final_iv_pred,19.4829
p99_final_iv_pred,31.8235
max_final_iv_pred,33.9346


In [75]:
print("06.2 final route mix")
display(
    final_test_predictions["hybrid_route"]
    .value_counts(dropna=False)
    .rename_axis("hybrid_route")
    .to_frame("count")
)


06.2 final route mix


,count
hybrid_route,
structured_winner,1071
histgb_pruned_v1_no_anchor_no_regime,583
ridge_direct_full,45


In [76]:
print("06.2 final structured source mix")
display(
    final_test_predictions["structured_winner_source"]
    .value_counts(dropna=False)
    .rename_axis("structured_winner_source")
    .to_frame("count")
)


06.2 final structured source mix


,count
structured_winner_source,
structured_region_blend_callput_shrink,971
quadratic_smile_only,408
same_date_linear_interp,180
structured_region_blend_center,68
tv_maturity_only,51
structured_region_blend_wing,21


In [77]:
final_vs_structured_wing_center = (
    final_test_predictions
    .groupby("wing_center_bucket", as_index=False)
    .agg(
        n=("row_id", "size"),
        mean_structured_iv=("structured_winner_pred", "mean"),
        mean_final_iv=("final_iv_pred", "mean"),
        mean_delta_vs_structured=("delta_vs_structured", "mean"),
        mean_abs_delta_vs_structured=("abs_delta_vs_structured", "mean"),
    )
    .sort_values("wing_center_bucket")
)

print("06.2 final vs structured | wing vs center")
display(final_vs_structured_wing_center)


06.2 final vs structured | wing vs center


,wing_center_bucket,n,mean_structured_iv,mean_final_iv,mean_delta_vs_structured,mean_abs_delta_vs_structured
0,center,923,20.2430,20.2430,0.0000,0.0000
1,wing,776,20.8326,21.0619,0.2293,0.5803


In [78]:
final_vs_structured_maturity = (
    final_test_predictions
    .groupby("maturity_label", as_index=False)
    .agg(
        n=("row_id", "size"),
        mean_structured_iv=("structured_winner_pred", "mean"),
        mean_final_iv=("final_iv_pred", "mean"),
        mean_delta_vs_structured=("delta_vs_structured", "mean"),
        mean_abs_delta_vs_structured=("abs_delta_vs_structured", "mean"),
    )
    .sort_values("maturity_label")
)

print("06.2 final vs structured | maturity")
display(final_vs_structured_maturity)


06.2 final vs structured | maturity


,maturity_label,n,mean_structured_iv,mean_final_iv,mean_delta_vs_structured,mean_abs_delta_vs_structured
0,1M,486,21.7131,21.8653,0.1522,0.3345
1,2M,416,20.7797,20.9722,0.1925,0.3426
2,3M,399,20.0846,20.1240,0.0394,0.2148
3,6M,398,19.1952,19.2158,0.0206,0.1495


In [79]:
final_total_variance_sanity = (
    final_test_predictions
    .groupby("maturity_label", as_index=False)
    .agg(
        n=("row_id", "size"),
        min_total_variance=("pred_total_variance", "min"),
        median_total_variance=("pred_total_variance", "median"),
        max_total_variance=("pred_total_variance", "max"),
    )
    .sort_values("maturity_label")
)

print("06.2 total-variance sanity by maturity")
display(final_total_variance_sanity)


06.2 total-variance sanity by maturity


,maturity_label,n,min_total_variance,median_total_variance,max_total_variance
0,1M,486,0.0020,0.0053,0.0137
1,2M,416,0.0039,0.0095,0.0257
2,3M,399,0.0060,0.0130,0.0364
3,6M,398,0.0129,0.0247,0.0548


In [80]:
final_submission_df = final_test_predictions[["row_id", "final_iv_pred"]].rename(
    columns={"final_iv_pred": "iv_pred"}
).sort_values("row_id").reset_index(drop=True)

print("06.2 final submission preview")
display(final_submission_df.head())
print("Number of submission rows:", len(final_submission_df))


06.2 final submission preview


,row_id,iv_pred
0,11643,22.4697
1,11645,21.2880
2,11648,17.5375
3,11650,16.7037
4,11651,18.0317


Number of submission rows: 1699


## Phase 6.2 final inference and submission checks: summary

This section applied the final locked model from Phase 6.1 to the true missing rows in the test set and ran final sanity checks before submission preparation.

The locked final model used here is:

- `hybrid_slice_no_hard_case_override_pruned`

with:

- shared training policy:
  - `primary_only`
- Ridge branch:
  - `alpha = 10.0`
- pruned HistGB branch:
  - `learning_rate = 0.05`
  - `max_depth = 3`
  - `max_iter = 300`
  - `min_samples_leaf = 10`

---

## 1. Final training and inference setup

The final pseudo-training table was rebuilt using the locked shared policy on the full training period.

Summary:
- pseudo-training rows: `592`
- pseudo-training dates: `77`
- protocol mix:
  - `primary_realistic` only

The actual test inference table was then built for the true missing rows only.

Summary:
- total test rows: `3960`
- true missing test rows to predict: `1699`
- test dates covered: `33`

All core structured and historical fallback features were available for those rows:
- `structured_winner_pred` missing: `0`
- `recent_node_pred` missing: `0`
- `full_node_pred` missing: `0`

---

## 2. Final prediction behavior

The final hybrid produced predictions for all missing rows.

Prediction range sanity:
- number of predictions: `1699`
- nonpositive predictions: `0`
- minimum predicted IV: `12.7884`
- 1st percentile predicted IV: `13.6089`
- median predicted IV: `19.4829`
- 99th percentile predicted IV: `31.8235`
- maximum predicted IV: `33.9346`

This range looks reasonable and does not show obvious instability.

---

## 3. Final route mix

The final hybrid routing on the true missing test rows was:

- `structured_winner = 1071`
- `histgb_pruned_v1_no_anchor_no_regime = 583`
- `ridge_direct_full = 45`

This is consistent with the validation-stage behavior:
- the structured branch remains the dominant route
- the pruned HistGB branch is meaningfully used on a large minority of rows
- the Ridge override remains narrow and targeted

The structured source mix is also coherent, with most rows coming from:
- `structured_region_blend_callput_shrink`
- `quadratic_smile_only`
- `same_date_linear_interp`

and only a small minority from:
- `tv_maturity_only`
- `structured_region_blend_center`
- `structured_region_blend_wing`

---

## 4. Final prediction vs structured baseline

The final model behaves exactly as intended relative to the structured baseline.

### Wing vs center
- center:
  - mean structured IV = `20.2430`
  - mean final IV = `20.2430`
  - mean delta vs structured = `0.0000`
- wing:
  - mean structured IV = `20.8326`
  - mean final IV = `21.0619`
  - mean delta vs structured = `0.2293`

This matches the locked routing logic:
- center is routed to the structured branch
- wing regions are where the nonlinear branch adds most of the hybrid adjustment

### By maturity
Average uplift vs structured:
- `1M = +0.1522`
- `2M = +0.1925`
- `3M = +0.0394`
- `6M = +0.0206`

So the hybrid adds the largest adjustments at shorter maturities and smaller adjustments at longer maturities, which is directionally sensible.

---

## 5. Total-variance sanity

Predicted total variance remains positive and the maturity-level medians increase in a clean way:

- `1M median total variance = 0.0053`
- `2M median total variance = 0.0095`
- `3M median total variance = 0.0130`
- `6M median total variance = 0.0247`

This is a useful final sanity check that the output surface is not obviously pathological by tenor.

---

## 6. Final submission artifact

The final submission-style table was created as:

- `row_id`
- `iv_pred`

with:

- `1699` rows

This matches the number of true missing test rows exactly.

---

## 7. Final conclusion

The Phase 6 workflow is now complete.

The final selected model for submission is:

- `hybrid_slice_no_hard_case_override_pruned`

using:

- shared training policy:
  - `primary_only`
- Ridge:
  - `alpha = 10.0`
- pruned HistGB:
  - `learning_rate = 0.05`
  - `max_depth = 3`
  - `max_iter = 300`
  - `min_samples_leaf = 10`

The final inference checks do not show any obvious operational issues:
- all required predictions were generated
- prediction ranges are reasonable
- route mix is coherent
- the hybrid behaves as expected relative to the structured baseline
- total-variance sanity is acceptable by maturity

So the project is ready to move from model-selection notebooks to final submission packaging using this locked model.


## Final artifact and consistency patch

This patch closes the remaining competition-packaging gaps.

It does three things:

1. builds the final submission in the exact competition format using `sample_submission.csv`
2. adds explicit integrity assertions on row alignment and prediction completeness
3. adds a nodewise calendar-consistency check on the completed test surface

Important clarification:
- the final ML branches are trained on the locked pseudo-supervised table induced by the `primary_only` policy
- this final pseudo-training table has `592` rows across `77` target dates
- test-time feature construction still uses the full `97` training dates as historical context


In [81]:
completed_test = test.copy()

completed_test = completed_test.merge(
    final_submission_df.rename(columns={"iv_pred": "iv_predicted_tmp"}),
    on="row_id",
    how="left",
)

completed_test["iv_completed"] = completed_test["iv_observed"]
fill_mask = completed_test["iv_completed"].isna()
completed_test.loc[fill_mask, "iv_completed"] = completed_test.loc[fill_mask, "iv_predicted_tmp"]

missing_after_completion = int(completed_test["iv_completed"].isna().sum())
print("Missing IVs after completion:", missing_after_completion)

if missing_after_completion != 0:
    raise ValueError(f"Completed test surface still has {missing_after_completion} missing IV values.")


Missing IVs after completion: 0


In [82]:
sample_submission = pd.read_csv(PROJECT_ROOT / "sample_submission.csv")

final_submission_df = (
    sample_submission[["row_id"]]
    .merge(
        final_test_predictions[["row_id", "final_iv_pred"]].rename(
            columns={"final_iv_pred": "iv_predicted"}
        ),
        on="row_id",
        how="left",
    )
)

print("06.2 final submission exact-format preview")
display(final_submission_df.head())
print("Number of submission rows:", len(final_submission_df))


06.2 final submission exact-format preview


,row_id,iv_predicted
0,11643,22.4697
1,11645,21.2880
2,11648,17.5375
3,11650,16.7037
4,11651,18.0317


Number of submission rows: 1699


In [83]:
print("06.2 submission integrity checks")

assert list(final_submission_df.columns) == ["row_id", "iv_predicted"], (
    f"Unexpected submission columns: {final_submission_df.columns.tolist()}"
)

assert len(final_submission_df) == len(sample_submission), (
    f"Row count mismatch: submission={len(final_submission_df)}, sample={len(sample_submission)}"
)

assert final_submission_df["row_id"].is_unique, "Submission row_id values are not unique."
assert sample_submission["row_id"].is_unique, "Sample submission row_id values are not unique."

assert final_submission_df["row_id"].tolist() == sample_submission["row_id"].tolist(), (
    "Submission row_id sequence does not exactly match sample_submission.csv"
)

assert final_submission_df["iv_predicted"].notna().all(), "Submission contains missing predictions."
assert (final_submission_df["iv_predicted"] > 0).all(), "Submission contains nonpositive predictions."

print("All submission integrity checks passed.")
display(
    pd.Series(
        {
            "n_rows": len(final_submission_df),
            "n_missing_predictions": int(final_submission_df["iv_predicted"].isna().sum()),
            "min_iv_predicted": float(final_submission_df["iv_predicted"].min()),
            "p01_iv_predicted": float(final_submission_df["iv_predicted"].quantile(0.01)),
            "median_iv_predicted": float(final_submission_df["iv_predicted"].median()),
            "p99_iv_predicted": float(final_submission_df["iv_predicted"].quantile(0.99)),
            "max_iv_predicted": float(final_submission_df["iv_predicted"].max()),
        }
    ).to_frame("value")
)


06.2 submission integrity checks
All submission integrity checks passed.


,value
n_rows,"1,699.0000"
n_missing_predictions,0.0000
min_iv_predicted,12.7884
p01_iv_predicted,13.6089
median_iv_predicted,19.4829
p99_iv_predicted,31.8235
max_iv_predicted,33.9346


In [84]:
soft_floor = 5.0
n_below_soft_floor = int((final_submission_df["iv_predicted"] < soft_floor).sum())
print(f"Predictions below soft floor {soft_floor}: {n_below_soft_floor}")


Predictions below soft floor 5.0: 0


In [85]:
calendar_check_df = completed_test[
    [
        "row_id",
        "date",
        "option_type",
        "moneyness",
        "maturity_label",
        "maturity_days",
        "tau",
        "iv_completed",
    ]
].copy()

calendar_check_df["total_variance"] = (
    (calendar_check_df["iv_completed"] / 100.0) ** 2 * calendar_check_df["tau"]
)

calendar_check_df = calendar_check_df.sort_values(
    ["date", "option_type", "moneyness", "maturity_days"]
).reset_index(drop=True)


In [88]:
calendar_violation_rows = []

for key, g in calendar_check_df.groupby(["date", "option_type", "moneyness"], sort=False):
    g = g.sort_values("maturity_days").copy()

    if len(g) < 2:
        continue

    tv = g["total_variance"].to_numpy()
    maturity_days = g["maturity_days"].to_numpy()
    row_ids = g["row_id"].to_numpy()

    diffs = np.diff(tv)
    for i, d_tv in enumerate(diffs):
        if d_tv < 0:
            calendar_violation_rows.append(
                {
                    "date": key[0],
                    "option_type": key[1],
                    "moneyness": key[2],
                    "row_id_left": int(row_ids[i]),
                    "row_id_right": int(row_ids[i + 1]),
                    "maturity_left": int(maturity_days[i]),
                    "maturity_right": int(maturity_days[i + 1]),
                    "tv_left": float(tv[i]),
                    "tv_right": float(tv[i + 1]),
                    "delta_total_variance": float(d_tv),
                }
            )

calendar_violations = pd.DataFrame(calendar_violation_rows)

print("06.2 nodewise calendar violation preview")
display(calendar_violations.head())


06.2 nodewise calendar violation preview


""


In [89]:
calendar_node_counts = (
    calendar_check_df.groupby(["date", "option_type", "moneyness"])
    .size()
    .rename("n_maturities")
    .reset_index()
)

n_nodes_checked = int((calendar_node_counts["n_maturities"] >= 2).sum())

if len(calendar_violations) > 0:
    violating_nodes = (
        calendar_violations[["date", "option_type", "moneyness"]]
        .drop_duplicates()
        .shape[0]
    )
else:
    violating_nodes = 0

calendar_violation_summary = pd.Series(
    {
        "n_nodes_checked": n_nodes_checked,
        "n_pairwise_calendar_violations": int(len(calendar_violations)),
        "n_violating_nodes": int(violating_nodes),
        "share_violating_nodes": float(violating_nodes / n_nodes_checked) if n_nodes_checked > 0 else np.nan,
    }
)

print("06.2 nodewise calendar violation summary")
display(calendar_violation_summary.to_frame("value"))


06.2 nodewise calendar violation summary


,value
n_nodes_checked,990.0000
n_pairwise_calendar_violations,0.0000
n_violating_nodes,0.0000
share_violating_nodes,0.0000


In [90]:
if len(calendar_violations) > 0:
    calendar_violation_severity = pd.Series(
        {
            "most_negative_delta_total_variance": float(calendar_violations["delta_total_variance"].min()),
            "median_negative_delta_total_variance": float(calendar_violations["delta_total_variance"].median()),
            "p01_negative_delta_total_variance": float(calendar_violations["delta_total_variance"].quantile(0.01)),
        }
    )
    print("06.2 calendar violation severity")
    display(calendar_violation_severity.to_frame("value"))
else:
    print("06.2 calendar violation severity")
    print("No nodewise calendar violations found.")


06.2 calendar violation severity
No nodewise calendar violations found.


In [91]:
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

final_submission_path = ARTIFACTS_DIR / "submission.csv"
final_submission_df.to_csv(final_submission_path, index=False)

print("Final submission written to:", final_submission_path)


Final submission written to: /Users/sauravkrishna/Documents/projects/NQFO-Impilied-volatility-surface/artifacts/submission.csv


In [92]:
final_model_provenance = pd.DataFrame(
    [
        {
            "field": "final_model_name",
            "value": "hybrid_slice_no_hard_case_override_pruned",
        },
        {
            "field": "shared_training_policy",
            "value": "primary_only",
        },
        {
            "field": "ridge_alpha",
            "value": "10.0",
        },
        {
            "field": "histgb_learning_rate",
            "value": "0.05",
        },
        {
            "field": "histgb_max_depth",
            "value": "3",
        },
        {
            "field": "histgb_max_iter",
            "value": "300",
        },
        {
            "field": "histgb_min_samples_leaf",
            "value": "10",
        },
        {
            "field": "submission_path",
            "value": str(final_submission_path),
        },
    ]
)

print("06.2 final model provenance")
display(final_model_provenance)


06.2 final model provenance


,field,value
0,final_model_name,hybrid_slice_no_hard_case_override_pruned
1,shared_training_policy,primary_only
2,ridge_alpha,10.0
3,histgb_learning_rate,0.05
4,histgb_max_depth,3
5,histgb_max_iter,300
6,histgb_min_samples_leaf,10
7,submission_path,/Users/sauravkrishna/Documents/projects/NQFO-I...
